# Upswing-Labs — Telegraph Season II Track 03 GPU LoRA Champion Forge

**1-Click Google Colab GPU Pipeline (`A100` / `L4` / `T4`)** for training, verifying, and packaging **Rank #1 Champion `modernbert-base-lora16-v1` `.safetensors` LoRA Evaluator Adapters** for Telegraph Protocol Season II Track 03 (`$300` Delivery + `$500` Excellence = `$800`, closing Dec 1, 2026).

### Live-Verified Championless Target Families (`devnode.telegraphprotocol.com`, Epoch 47)
| Family (`register_on`) | Class | Eligible Submitters | Eligible Rows | Current Champion | Strategic Synergy |
|---|---|---:|---:|---|---|
| **`LIQUIDITY_DEPTH_VERIFY`** *(default)* | `judgment` | **8 / 3** | **527** (275 qual / 252 rank) | `null` | Missions 01/02/03/09/13/14 + Track 02 Miner |
| **`MACRO_ECONOMIC_INDICATOR`** | `judgment` | **6 / 3** | **531** (268 qual / 263 rank) | `null` | Missions 03/05/06/08/13 + Track 01 FactGap |
| **`FACT_CHECK`** | `judgment` | **4 / 3** | **474** (245 qual / 229 rank) | `null` | Missions 05/07 + Track 01 FactGap |

### Pinned Official Foundation Contract (`telegraphprotocol/telegraph-foundation` @ `modernbert-base-lora16-v1`)
- **Base Backbone**: `answerdotai/ModernBERT-base` @ `8949b909ec900327062f0ebf497f51aef5e6f0c8`
- **Foundation ONNX SHA-256**: `0a49b325304bb4cf292e683929689d0457966be264f60bfc94d0072b614bf8e6`
- **Foundation Engine WASM SHA-256**: `77643d6e1176ab8744801cef93ccb844d0136ed32b8daa865d541f37d13292de`
- **Pinned Tokenizer SHA-256**: `67bc76f8b9386efba1a761c1c8e5ad439fd0eaf1c2826e9dd1a205c7af4dc456` *(auto-fixes upstream Issue `#223` HF mismatch `9fd55248...` + `train_adapter.py:838` `NameError`!)*
- **Delta Package Contract (`lora-v1`)**: Exact **178 F32 tensors** tiled contiguously (`lora.0..21.{attn.Wqkv,attn.Wo,mlp.Wi,mlp.Wo}.{A,B}` + `head.weight [768,1]` + `head.bias [1]`), `r <= 16`, `max_tokens <= 512`, `<= 64 MiB`.

> **How to run:** `Runtime -> Change runtime type -> T4 / L4 / A100 GPU`, then click **`Runtime -> Run all`**. When Cell 6 completes, your verified `<INTENT>_upswing_champion.safetensors`, `training_evidence.json`, and `adapterexec_deploy` bundle auto-download to your PC.


## Cell 1 — Configuration & GPU Hardware Auto-Tuning (`A100` / `L4` / `T4`)
Configure which championless families to train (default: `LIQUIDITY_DEPTH_VERIFY`; set `TRAIN_PRESET = "ALL_THREE_CHAMPIONS"` to train all 3 in one ~20–25 min GPU session). Automatically detects VRAM & `bf16` support and disables Triton/Inductor compilation so ModernBERT runs smoothly on Colab T4 (`sm_75`) as well as L4/A100.


In [ ]:
# @title Upswing-Labs Track 03 Champion Configuration & GPU Auto-Tuning
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

# Prevent ModernBERT torch.compile / Triton failures on Colab T4 (sm_75)
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ---- User Controls (Colab Form or direct edit) ----
TRAIN_PRESET = "LIQUIDITY_ONLY"  # @param ["LIQUIDITY_ONLY", "ALL_THREE_CHAMPIONS", "CUSTOM"]
# Override directly when TRAIN_PRESET == "CUSTOM":
TARGET_INTENTS = ["LIQUIDITY_DEPTH_VERIFY"]
if TRAIN_PRESET == "LIQUIDITY_ONLY":
    TARGET_INTENTS = ["LIQUIDITY_DEPTH_VERIFY"]
elif TRAIN_PRESET == "ALL_THREE_CHAMPIONS":
    TARGET_INTENTS = ["LIQUIDITY_DEPTH_VERIFY", "MACRO_ECONOMIC_INDICATOR", "FACT_CHECK"]

PAIRS_PER_INTENT = 600           # @param {type:"integer"}
RUN_HPARAM_SWEEP = True          # @param {type:"boolean"}
VERIFY_ON_WASM_FOUNDATION = False  # @param {type:"boolean"}
LAUNCH_COLAB_CLOUDFLARED_EXECUTOR = False  # @param {type:"boolean"}
AUTHOR_ORG = "Upswing-Labs"
RANDOM_SEED = 42

# Live-verified intent metadata (devnode.telegraphprotocol.com /engine/v1/intents/*, Epoch 47)
INTENT_SPECS = {
    "LIQUIDITY_DEPTH_VERIFY": {
        "family": "LIQUIDITY_DEPTH_VERIFY",
        "child_intent": "LIQUIDITY_DEPTH_VERIFY_GENERAL",
        "class": "judgment",
        "eligible_submitters": 8,
        "eligible_rows": 527,
        "missions": ["01", "02", "03", "09", "13", "14"],
    },
    "MACRO_ECONOMIC_INDICATOR": {
        "family": "MACRO_ECONOMIC_INDICATOR",
        "child_intent": "MACRO_ECONOMIC_INDICATOR_GENERAL",
        "class": "judgment",
        "eligible_submitters": 6,
        "eligible_rows": 531,
        "missions": ["03", "05", "06", "08", "13"],
    },
    "FACT_CHECK": {
        "family": "FACT_CHECK",
        "child_intent": "FACT_CHECK_GENERAL",
        "class": "judgment",
        "eligible_submitters": 4,
        "eligible_rows": 474,
        "missions": ["05", "07"],
    },
}

for _t in TARGET_INTENTS:
    if _t not in INTENT_SPECS:
        raise ValueError(f"Unsupported target intent {_t!r}; choose from {list(INTENT_SPECS)}")

WORK_ROOT = Path("/content/work") if Path("/content").exists() else Path("./upswing_colab_work").resolve()
KIT_DIR = WORK_ROOT / "telegraph-wasm-baseline"
FOUNDATION_DIR = WORK_ROOT / "foundation"
PINNED_MODEL_DIR = WORK_ROOT / "modernbert_base_pinned"
DATA_DIR = WORK_ROOT / "datasets"
RUNS_DIR = WORK_ROOT / "runs"
EXPORT_DIR = WORK_ROOT / "exports"
for _d in (WORK_ROOT, FOUNDATION_DIR, DATA_DIR, RUNS_DIR, EXPORT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Detect GPU via nvidia-smi + PyTorch
print("=== Hardware & GPU Inspection ===")
try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    )
    print(smi.stdout.strip() or "nvidia-smi returned empty output")
except FileNotFoundError:
    print("nvidia-smi not found (CPU environment)")

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram_gb = round(props.total_memory / (1024 ** 3), 2)
    cc = f"{props.major}.{props.minor}"
    bf16_ok = bool(torch.cuda.is_bf16_supported())
    if vram_gb >= 35:  # A100 40/80GB or H100
        tier = "A100/H100"
        batch_size, grad_accum, eval_batch, grad_ckpt = 8, 2, 16, False
    elif vram_gb >= 20:  # L4 24GB or A10G
        tier = "L4/24GB"
        batch_size, grad_accum, eval_batch, grad_ckpt = 4, 4, 8, True
    else:  # T4 15-16GB or V100
        tier = "T4/16GB"
        batch_size, grad_accum, eval_batch, grad_ckpt = 2, 8, 4, True
    precision = "bf16" if bf16_ok else "fp16"
    device_str = "cuda"
else:
    gpu_name, vram_gb, cc, bf16_ok = "CPU-only", 0.0, "n/a", False
    tier, batch_size, grad_accum, eval_batch, grad_ckpt = "CPU", 1, 16, 4, True
    precision, device_str = "fp32", "cpu"
    print("WARNING: No CUDA GPU detected! In Colab go to Runtime -> Change runtime type -> T4/L4/A100 GPU.")

GPU_PROFILE = {
    "device": device_str,
    "gpu_name": gpu_name,
    "tier": tier,
    "vram_gb": vram_gb,
    "compute_capability": cc,
    "precision": precision,
    "bf16_supported": bf16_ok,
    "max_length": 512,
    "batch_size": batch_size,
    "grad_accum": grad_accum,
    "effective_batch_size": batch_size * grad_accum,
    "eval_batch": eval_batch,
    "grad_checkpointing": grad_ckpt,
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
}

print("\n=== Auto-Tuned Training Profile ===")
print(json.dumps(GPU_PROFILE, indent=2))
print(f"\nTarget Championless Families ({len(TARGET_INTENTS)}): {TARGET_INTENTS}")
print(f"Pairs/Intent: {PAIRS_PER_INTENT} | Sweep: {RUN_HPARAM_SWEEP} | WASM Double-Score Check: {VERIFY_ON_WASM_FOUNDATION}")


## Cell 2 — Clone Official `telegraph-wasm-baseline`, Download Pinned `modernbert-base-lora16-v1` Assets & Auto-Fix Bug `#223`
1. Installs pinned `transformers`, `peft`, `safetensors`, `huggingface_hub`, and `pycryptodome` (for true Ethereum Keccak-256 `wasmHash`).
2. Clones `https://github.com/telegraphprotocol/telegraph-wasm-baseline` and automatically patches **Bug `#223` Part B** (`float_runtime/train_adapter.py` line 838 `NameError`: `{pkg}` -> `{deltapkg}`).
3. Downloads the exact pinned release assets (`foundation.json`, `tokenizer.json`, `SHA256SUMS`) from `telegraphprotocol/telegraph-foundation` tag `modernbert-base-lora16-v1` and verifies every SHA-256 digest.
4. Applies the verified **Bug `#223` Part A** fix: downloads `answerdotai/ModernBERT-base` @ revision `8949b909ec900327062f0ebf497f51aef5e6f0c8`, replaces Hugging Face's `tokenizer.json` (`9fd55248...`) with Telegraph's pinned release `tokenizer.json` (`67bc76f8...`), and passes all **88/88** official conformance vectors (`tokenizer_vectors.json`).


In [ ]:
# @title Install Dependencies, Clone Official Adapter Kit & Verify Pinned Foundation (Bug #223 Auto-Fix)
import hashlib
import json
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

# 1. Install pinned compatible packages
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "transformers>=4.48.3,<5.0.0",
        "peft>=0.14.0,<0.18.0",
        "safetensors>=0.4.5",
        "huggingface_hub>=0.27.0",
        "pycryptodome>=3.20.0",
        "numpy",
        "pyyaml",
    ],
    check=True,
)

# 2. Verify true Ethereum Keccak-256 (NOT NIST SHA3-256!)
from Crypto.Hash import keccak


def keccak256_hex(data: bytes, prefix: bool = True) -> str:
    h = keccak.new(digest_bits=256, data=data).hexdigest()
    return f"0x{h}" if prefix else h


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


assert keccak256_hex(b"", prefix=False) == "c5d2460186f7233c927e7db2dcc703c0e500b653ca82273b7bfad8045d85a470", (
    "CRITICAL: Keccak-256 self-test failed!"
)
print("OK: Ethereum Keccak-256 self-test passed.")

# 3. Clone official telegraphprotocol/telegraph-wasm-baseline
KIT_REPO = "https://github.com/telegraphprotocol/telegraph-wasm-baseline.git"
if not (KIT_DIR / "float_runtime" / "train_adapter.py").exists():
    if KIT_DIR.exists():
        shutil.rmtree(KIT_DIR)
    subprocess.run(["git", "clone", "--depth", "1", KIT_REPO, str(KIT_DIR)], check=True)
else:
    print(f"Reusing existing checkout at {KIT_DIR}")

kit_commit = subprocess.run(
    ["git", "-C", str(KIT_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print(f"Official telegraph-wasm-baseline commit: {kit_commit}")

# Automated Bug #223 Part B fix: train_adapter.py line 838 prints {pkg} instead of {deltapkg}
ta_path = KIT_DIR / "float_runtime" / "train_adapter.py"
ta_src = ta_path.read_text(encoding="utf-8")
if "--package {pkg}" in ta_src:
    ta_path.write_text(ta_src.replace("--package {pkg}", "--package {deltapkg}"), encoding="utf-8")
    print("PATCHED Bug #223 Part B: fixed NameError ({pkg} -> {deltapkg}) in train_adapter.py")
else:
    print("OK: train_adapter.py already clean of Bug #223 NameError.")

if str(KIT_DIR) not in sys.path:
    sys.path.insert(0, str(KIT_DIR))

# 4. Download & SHA-256 verify exact pinned modernbert-base-lora16-v1 release assets
FOUNDATION_TAG = "modernbert-base-lora16-v1"
FOUNDATION_BASE_URL = f"https://github.com/telegraphprotocol/telegraph-foundation/releases/download/{FOUNDATION_TAG}"
PINNED_HASHES = {
    "foundation.json": "f7de64b999acd3f735118fbe5999abf251fdf188f9797e3a4e36c54ee9e4dbbe",
    "tokenizer.json": "67bc76f8b9386efba1a761c1c8e5ad439fd0eaf1c2826e9dd1a205c7af4dc456",
    "foundation_lora_fp32.onnx": "0a49b325304bb4cf292e683929689d0457966be264f60bfc94d0072b614bf8e6",
    "foundation_engine.wasm": "77643d6e1176ab8744801cef93ccb844d0136ed32b8daa865d541f37d13292de",
}


def download_verified(fname: str, expected_sha: str | None = None) -> Path:
    dest = FOUNDATION_DIR / fname
    if not dest.exists() or (expected_sha and sha256_file(dest) != expected_sha):
        url = f"{FOUNDATION_BASE_URL}/{fname}"
        print(f"Downloading {url} -> {dest.name} ...")
        req = urllib.request.Request(url, headers={"User-Agent": "Upswing-Labs-Colab-Forge"})
        with urllib.request.urlopen(req, timeout=120) as resp, dest.open("wb") as out_fh:
            shutil.copyfileobj(resp, out_fh)
    actual = sha256_file(dest)
    if expected_sha and actual != expected_sha:
        raise RuntimeError(f"SHA-256 mismatch for {fname}: got {actual}, expected {expected_sha}")
    print(f"  VERIFIED {fname}: sha256={actual} ({dest.stat().st_size:,} bytes)")
    return dest


FOUNDATION_JSON_PATH = download_verified("foundation.json", PINNED_HASHES["foundation.json"])
PINNED_TOKENIZER_PATH = download_verified("tokenizer.json", PINNED_HASHES["tokenizer.json"])
download_verified("SHA256SUMS", None)

FOUNDATION_SPEC = json.loads(FOUNDATION_JSON_PATH.read_text(encoding="utf-8"))
assert FOUNDATION_SPEC["foundation_version"] == FOUNDATION_TAG
assert FOUNDATION_SPEC["foundation_hash"] == PINNED_HASHES["foundation_lora_fp32.onnx"]
assert FOUNDATION_SPEC["tokenizer_hash"] == PINNED_HASHES["tokenizer.json"]
print("OK: foundation.json matches canonical modernbert-base-lora16-v1 hashes.")

# 5. Automated Bug #223 Part A fix: local ModernBERT-base snapshot with pinned release tokenizer.json
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

BASE_MODEL_ID = "answerdotai/ModernBERT-base"
BASE_MODEL_REV = "8949b909ec900327062f0ebf497f51aef5e6f0c8"

if not (PINNED_MODEL_DIR / "config.json").exists():
    print(f"\nDownloading {BASE_MODEL_ID} @ {BASE_MODEL_REV[:12]} ...")
    snapshot_download(
        repo_id=BASE_MODEL_ID,
        revision=BASE_MODEL_REV,
        local_dir=str(PINNED_MODEL_DIR),
        ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*"],
    )

hf_tok_path = PINNED_MODEL_DIR / "tokenizer.json"
hf_tok_sha = sha256_file(hf_tok_path) if hf_tok_path.exists() else "missing"
print(f"HF snapshot tokenizer.json sha256 before Bug #223 fix: {hf_tok_sha}")
shutil.copy2(PINNED_TOKENIZER_PATH, hf_tok_path)
pinned_sha = sha256_file(hf_tok_path)
assert pinned_sha == PINNED_HASHES["tokenizer.json"], f"Failed to pin tokenizer.json: {pinned_sha}"
print(f"Pinned local snapshot tokenizer.json sha256:           {pinned_sha}")

# Verify 88/88 official Telegraph tokenizer conformance vectors
vectors_path = KIT_DIR / "float_runtime" / "testdata" / "tokenizer_vectors.json"
tok_check = AutoTokenizer.from_pretrained(str(PINNED_MODEL_DIR))
vectors = json.loads(vectors_path.read_text(encoding="utf-8"))["vectors"]
for v in vectors:
    got_ids = list(tok_check(v["text"], add_special_tokens=v["add_special_tokens"])["input_ids"])
    if got_ids != list(v["ids"]):
        raise RuntimeError(f"Tokenizer vector mismatch on {v['text'][:40]!r}: {got_ids[:10]} != {v['ids'][:10]}")
print(f"PASS: Pinned ModernBERT tokenizer matched {len(vectors)}/{len(vectors)} official node conformance vectors!")


## Cell 3 — Build Leak-Free, Shortcut-Free 600-Pair Domain Corpora & Pass All 6 Telegraph Dataset Gates
Generates **600 genuinely distinct preference pairs** per target family with:
- **100% unique rejected answers** (`>= 80%` required) & **0.17% max single negative** (`<= 2%` required)
- **~50.0% length balance** (`<= 65%` required) & **0.0% held-out negative leakage** (`<= 10%` required)
- **Bijective contrastive symmetry** -> prompt-blind 5-seed bag-of-words probe stays **~40–50%** (`<= 65%` required), forcing ModernBERT to learn **true cross-attention domain verification** (`### Task:` vs `### Answer:`)!
- **All real failure modes**: Bug `#207` cross-chain/wrong-pool substitution (BNB/ETH/Solana), reserve-sum vs USD accounting mismatch, FDV-confused-as-TVL trap, zero-liquidity honeypot vs deep-pool regime verification, constant-product AMM slippage physics, multi-format JSON/prose/key-value answers!


In [ ]:
# @title Build 600-Pair Zero-Shortcut Corpora, Verify Official Telegraph Dataset Gate & Split 85/15
import hashlib
import json
import random
from collections import Counter
from pathlib import Path

from float_runtime.train_adapter import Hyperparams, format_gate, gate_dataset, split_pairs


def _balance_pair_length(chosen: str, rejected: str, idx: int, tail_c: str, tail_r: str) -> tuple[str, str]:
    want_chosen_longer = (idx % 2 == 0)
    if want_chosen_longer and len(chosen) <= len(rejected):
        chosen = chosen + tail_c
    elif (not want_chosen_longer) and len(rejected) <= len(chosen):
        rejected = rejected + tail_r
    return chosen, rejected


def _make_pool_addr(chain: str, seed_int: int) -> str:
    h = hashlib.sha256(f"upswing-pool:{chain}:{seed_int}".encode()).hexdigest()
    if chain == "solana":
        alphabet = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"
        val = int(h, 16)
        chars = []
        while len(chars) < 44:
            val, rem = divmod(val, 58)
            chars.append(alphabet[rem])
            if val == 0:
                val = int(hashlib.sha256(f"{h}:{len(chars)}".encode()).hexdigest(), 16)
        return "".join(chars)
    return "0x" + h[:40]


def build_liquidity_corpus(n_pairs: int = 600, seed: int = 42) -> list[dict]:
    rng = random.Random(seed)
    anchor_pools = [
        ("ethereum", "eth", "Uniswap v3", "WETH/USDC", "0.05%", "0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640", 142850000),
        ("ethereum", "eth", "Uniswap v3", "WBTC/WETH", "0.30%", "0xCBCdF9626bC03E24f779434178A73a0B4bad62eD", 118420000),
        ("ethereum", "eth", "Uniswap v2", "vensa/WETH", "0.30%", "0x1990D912C5F2Bf8638a5bCd4EAD66df9596bf851", 0),
        ("ethereum", "eth", "Curve", "stETH/ETH", "0.04%", "0xDC24316b9AE028F1497c275EB9192a3Ea0f67022", 245600000),
        ("solana", "solana", "Raydium CLMM", "SOL/USDC", "0.04%", "58oQChx4yWmvKdwLLZzBi4ChoCc2fqCUWBkwMihLYQo2", 64910000),
        ("solana", "solana", "Orca Whirlpool", "JUP/SOL", "0.05%", "C1MgLjobfozkK2WJgNcxMA4yB2J3H4a4yB4G4f8s4d3c", 22480000),
        ("base", "base", "Aerodrome Slipstream", "WETH/USDC", "0.05%", "0xb2cc224c1c9feE385f8ad6a55b4d94E92359DC59", 93640000),
        ("arbitrum", "arbitrum", "Uniswap v3", "WETH/USDC", "0.05%", "0xC6962004f452bE9203591991D15f6b388e09E8D0", 87350000),
        ("bsc", "bsc", "PancakeSwap v3", "ETH/USDC", "0.05%", "0x531FEbfeb9a61D948c384ACFBe6dCc51057AEa7e", 28000000),
        ("bsc", "bsc", "PancakeSwap v3", "WBNB/USDT", "0.05%", "0x36696169C63e42cd08ce11f5deeBbCeBae652050", 78540000),
        ("polygon", "polygon_pos", "QuickSwap", "WPOL/USDC", "0.25%", "0x6e7a5FAFcec6BB1e78bAE2A1F0B612012BF14827", 24310000),
        ("optimism", "optimism", "Velodrome", "OP/USDC", "0.05%", "0x47029bc8f5CBe3b464004E87eF9c9419a48018cd", 27860000),
        ("avalanche", "avax", "LFJ", "AVAX/USDC", "0.20%", "0xD446eb1660F766d533BeCeEf890Df7A69d26f7d1", 33400000),
    ]
    chains_cfg = [
        ("ethereum", "eth", ["Uniswap v3", "Uniswap v2", "Curve", "Balancer v2", "SushiSwap v3"]),
        ("solana", "solana", ["Raydium CLMM", "Raydium AMM", "Orca Whirlpool", "Meteora DLMM", "Lifinity"]),
        ("base", "base", ["Aerodrome Slipstream", "Aerodrome v2", "Uniswap v3", "BaseSwap", "PancakeSwap v3"]),
        ("arbitrum", "arbitrum", ["Uniswap v3", "Camelot v3", "SushiSwap", "Ramses", "Balancer"]),
        ("bsc", "bsc", ["PancakeSwap v3", "PancakeSwap v2", "Thena", "Biswap", "Uniswap v3"]),
        ("polygon", "polygon_pos", ["QuickSwap v3", "Uniswap v3", "Curve", "Retro", "Balancer"]),
        ("optimism", "optimism", ["Velodrome Slipstream", "Velodrome v2", "Uniswap v3", "Beethoven X", "Curve"]),
        ("avalanche", "avax", ["LFJ v2.1", "Pangolin", "Uniswap v3", "Pharaoh", "Curve"]),
    ]
    base_tokens = [
        "WETH", "WBTC", "SOL", "LINK", "UNI", "AAVE", "ARB", "OP", "AVAX", "WBNB", "WPOL", "AERO",
        "JUP", "RAY", "PYTH", "JTO", "BONK", "WIF", "PEPE", "SHIB", "LDO", "MKR", "CRV", "ENS",
        "SNX", "COMP", "GMX", "PENDLE", "ENA", "ONDO", "VIRTUAL", "BRETT", "DEGEN", "CAKE", "JOE",
        "VELO", "MAGIC", "GRAIL", "POPCAT", "RENDER", "HNT", "DRIFT", "KMNO", "TWT", "WELL", "GNO",
        "RPL", "FXS", "BAL", "SUSHI", "1INCH", "DYDX", "GRT", "IMX", "STG", "BLUR", "PRIME", "MORPHO",
        "EIGEN", "ETHFI", "SWELL", "ZRO", "STRK", "MANTA", "SEI", "TIA", "INJ", "NEAR", "SUI", "APT",
        "VENSA", "XTRAP", "HONEYX", "RUGLP", "DUSTX", "ZLP", "FAKEV2", "TAX99", "SCAMLP", "SHELLX",
    ]
    quote_tokens = ["USDC", "USDT", "WETH", "SOL", "DAI", "WBNB", "WBTC", "USDe", "crvUSD", "FRAX"]
    fee_tiers = ["0.01%", "0.04%", "0.05%", "0.20%", "0.25%", "0.30%", "1.00%"]

    pools = []
    for i in range(n_pairs):
        if i < len(anchor_pools):
            ch, net, dex, pair, fee, addr, liq = anchor_pools[i]
        else:
            ch, net, dexes = chains_cfg[i % len(chains_cfg)]
            dex = dexes[(i // len(chains_cfg)) % len(dexes)]
            bt = base_tokens[(i * 7 + (i // 8)) % len(base_tokens)]
            qt = quote_tokens[(i * 3 + (i // 11)) % len(quote_tokens)]
            if bt == qt:
                qt = "USDC" if bt != "USDC" else "USDT"
            pair = f"{bt}/{qt}"
            fee = fee_tiers[(i * 5 + (i // 7)) % len(fee_tiers)]
            addr = _make_pool_addr(ch, i)
            bucket = i % 20
            if bucket < 4:
                liq = 0 if (bucket == 0) else int(rng.randint(15, 480))
            elif bucket < 10:
                liq = int(round(rng.uniform(18_000, 340_000), -2))
            elif bucket < 15:
                liq = int(round(rng.uniform(1_200_000, 17_500_000), -3))
            else:
                liq = int(round(rng.uniform(22_000_000, 260_000_000), -4))
        split = 0.45 + ((i * 17) % 11) / 100.0
        base_r = int(round(liq * split))
        quote_r = liq - base_r
        trade_usd = [10000, 25000, 50000, 100000, 250000, 500000][i % 6]
        slip_bps = 9950.0 if liq <= 500 else round((trade_usd / (max(1.0, liq / 2.0) + trade_usd)) * 10000.0, 2)
        pools.append({
            "idx": i, "chain": ch, "net": net, "dex": dex, "pair": pair, "fee": fee,
            "addr": addr, "liq": liq, "base_r": base_r, "quote_r": quote_r,
            "trade_usd": trade_usd, "slip_bps": slip_bps,
        })

    by_mode = {m: [i for i in range(n_pairs) if (i % 5) == m] for m in range(5)}
    xchain_sorted = sorted(by_mode[0] + by_mode[1], key=lambda k: (pools[k]["chain"], k))
    shift_x = len(xchain_sorted) // 3 + 7
    xchain_partner = {xchain_sorted[p]: xchain_sorted[(p + shift_x) % len(xchain_sorted)] for p in range(len(xchain_sorted))}

    m2_sorted = sorted(by_mode[2], key=lambda k: pools[k]["liq"])
    m2_partner = {m2_sorted[p]: m2_sorted[len(m2_sorted) - 1 - p] for p in range(len(m2_sorted))}

    m3_sorted = sorted(by_mode[3], key=lambda k: pools[k]["liq"])
    m3_partner = {m3_sorted[p]: m3_sorted[len(m3_sorted) - 1 - p] for p in range(len(m3_sorted))}

    m4_sorted = sorted(by_mode[4], key=lambda k: pools[k]["slip_bps"])
    m4_partner = {m4_sorted[p]: m4_sorted[len(m4_sorted) - 1 - p] for p in range(len(m4_sorted))}

    prompt_templates = [
        lambda p, d: f"What is the total liquidity in USD of the {p['dex']} {p['pair']} ({p['fee']}) pool at address {p['addr']} on {p['net']} (chain id {p['chain']})? [ref {d}]",
        lambda p, d: f'Canonical ask LIQUIDITY_DEPTH_VERIFY_GENERAL: {{"pool":"{p["addr"]}","chain":"{p["chain"]}","dex":"{p["dex"]}","pair":"{p["pair"]}"}} — report reserves and ${p["trade_usd"]:,} impact ({d}).',
        lambda p, d: f"Verify current on-chain liquidity_usd and dual-sided reserves for {p['pair']} ({p['fee']}) hosted on {p['dex']} over {p['chain']} contract {p['addr']} (audit {d}).",
        lambda p, d: f"Pre-listing desk check ({d}): inspect {p['chain']} DEX pool {p['addr']} ({p['dex']} {p['pair']} fee {p['fee']}). Confirm total USD depth and ${p['trade_usd']:,} swap slippage.",
        lambda p, d: f"Measure active AMM reserve depth on {p['chain']} for pool={p['addr']} ({p['pair']} via {p['dex']}, tier {p['fee']}), batch {d}.",
        lambda p, d: f"Risk screen {d}: evaluate whether {p['dex']} {p['pair']} ({p['addr']} on {p['chain']}) has sufficient USD reserves for a ${p['trade_usd']:,} order.",
        lambda p, d: f"Settlement oracle lookup {d}: return verified liquidity_usd, base_reserve_usd, and quote_reserve_usd for {p['chain']} pool {p['addr']} ({p['dex']} {p['pair']}).",
        lambda p, d: f"Query live DEX pool reserves for {p['pair']} {p['fee']} at {p['addr']} on {p['net']}/{p['chain']} ({p['dex']}) — ticket {d}.",
        lambda p, d: f"Confirm exact USD liquidity depth of {p['addr']} ({p['dex']} {p['pair']} {p['fee']}, chain {p['chain']}) and expected bps slippage on ${p['trade_usd']:,} ({d}).",
        lambda p, d: f"Treasury execution check {d}: fetch current pool liquidity_usd on {p['chain']} for {p['dex']} contract {p['addr']} ({p['pair']} {p['fee']}).",
        lambda p, d: f"Routing verification {d}: validate dual-sided USD liquidity and {p['fee']} fee tier for {p['pair']} on {p['chain']} ({p['dex']} pool {p['addr']}).",
        lambda p, d: f"On-chain depth audit {d}: how much USD liquidity sits in {p['chain']} pool {p['addr']} ({p['dex']}, {p['pair']}, {p['fee']})?",
        lambda p, d: f"DEX depth telemetry {d}: verify {p['chain']} liquidity pool {p['addr']} ({p['pair']} {p['fee']} on {p['dex']}) reserve balance and slippage.",
    ]

    def render_ans(d: dict, fmt: int, trade_usd: int) -> str:
        if fmt == 0:
            return json.dumps({
                "pool": d["addr"], "dex": d["dex"], "pair": d["pair"], "fee_tier": d["fee"],
                "chain": d["chain"], "liquidity_usd": d["liq"],
                "base_reserve_usd": d["base_r"], "quote_reserve_usd": d["quote_r"],
                "slippage_bps": d["slip_bps"], "unit": "USD",
            })
        elif fmt == 1:
            return (
                f"Pool {d['dex']} {d['pair']} ({d['fee']}) at {d['addr']} on {d['chain']} holds "
                f"liquidity_usd={d['liq']} (base {d['base_r']} USD + quote {d['quote_r']} USD). "
                f"Estimated ${trade_usd:,} swap impact is {d['slip_bps']} bps."
            )
        elif fmt == 2:
            return (
                f'{{"liquidity_usd": {d["liq"]}, "chain": "{d["chain"]}", "pool": "{d["addr"]}"}} '
                f"— {d['dex']} {d['pair']} ({d['fee']}), reserves {d['base_r']}+{d['quote_r']} USD, "
                f"{d['slip_bps']} bps impact on ${trade_usd:,}."
            )
        else:
            return (
                f"Chain: {d['chain']} | Pool: {d['addr']} ({d['dex']} {d['pair']} {d['fee']}) | "
                f"liquidity_usd: {d['liq']} | Reserves: {d['base_r']}+{d['quote_r']} USD | "
                f"Impact(${trade_usd:,}): {d['slip_bps']} bps"
            )

    rows = []
    for i, p in enumerate(pools):
        desk_id = f"LDV-{10000 + i * 7}"
        prompt = prompt_templates[i % len(prompt_templates)](p, desk_id)
        mode = i % 5
        if mode in (0, 1):
            failure_tag = "cross_chain_pool_swap"
            bad = dict(pools[xchain_partner[i]])
        elif mode == 2:
            failure_tag = "honeypot_or_regime_inversion"
            donor = pools[m2_partner[i]]
            bad = dict(p, liq=donor["liq"], base_r=donor["base_r"], quote_r=donor["quote_r"], slip_bps=donor["slip_bps"])
            if p["liq"] <= 500:
                prompt += " On-chain quote balance reads dust (<0.001 quote units; sell simulation fails)."
            elif p["liq"] <= 400_000:
                prompt += f" On-chain indexer observes thin micro-pool reserves (~${p['liq'] // 1000}k total)."
            elif p["liq"] < 22_000_000:
                prompt += f" On-chain indexer observes mid-tier balanced reserves (~${p['liq'] / 1e6:.1f}M total)."
            else:
                prompt += f" On-chain indexer observes deep tier-1 institutional reserves (~${p['liq'] / 1e6:.1f}M total)."
        elif mode == 3:
            failure_tag = "reserve_accounting_fdv_mismatch"
            donor = pools[m3_partner[i]]
            bad = dict(p, liq=donor["liq"])
        else:
            failure_tag = "slippage_physics_violation"
            donor = pools[m4_partner[i]]
            bad = dict(p, slip_bps=donor["slip_bps"])

        fmt = i % 4
        chosen = render_ans(p, fmt, p["trade_usd"])
        rejected = render_ans(bad, fmt, p["trade_usd"])
        chosen, rejected = _balance_pair_length(
            chosen, rejected, i,
            f" Snapshot {p['chain']}:{desk_id}.",
            f" Snapshot {bad['chain']}:{desk_id}.",
        )
        rows.append({
            "intent_id": "LIQUIDITY_DEPTH_VERIFY_GENERAL",
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected,
            "label_kind": "preference_pair",
            "failure_mode": failure_tag,
        })
    return rows


MACRO_COUNTRIES = [
    ("United States", "USA", "North America", 4.0, 2.9, 2.8, 4.75),
    ("Germany", "DEU", "Eurozone", 3.4, 2.3, -0.2, 3.25),
    ("United Kingdom", "GBR", "Western Europe", 4.3, 2.5, 0.9, 4.75),
    ("France", "FRA", "Eurozone", 7.4, 2.0, 1.1, 3.25),
    ("Italy", "ITA", "Southern Europe", 6.5, 1.0, 0.7, 3.25),
    ("Spain", "ESP", "Iberia", 11.4, 2.8, 3.1, 3.25),
    ("Japan", "JPN", "East Asia", 2.5, 2.7, 0.1, 0.25),
    ("Canada", "CAN", "North America", 6.4, 2.4, 1.3, 3.75),
    ("Australia", "AUS", "Oceania", 4.1, 3.2, 1.2, 4.35),
    ("Brazil", "BRA", "South America", 6.8, 4.4, 3.2, 11.25),
    ("Mexico", "MEX", "Latin America", 2.7, 4.7, 1.5, 10.25),
    ("India", "IND", "South Asia", 4.2, 4.9, 6.8, 6.50),
    ("China", "CHN", "East Asia", 5.1, 0.3, 4.9, 3.10),
    ("South Korea", "KOR", "East Asia", 2.8, 2.3, 2.2, 3.25),
    ("Netherlands", "NLD", "Benelux", 3.7, 3.3, 0.9, 3.25),
    ("Switzerland", "CHE", "Alpine Europe", 2.4, 1.1, 1.3, 1.00),
    ("Sweden", "SWE", "Nordics", 8.4, 2.8, 0.8, 2.75),
    ("Norway", "NOR", "Nordics", 4.0, 3.1, 1.5, 4.50),
    ("Poland", "POL", "Central Europe", 2.9, 3.7, 2.9, 5.75),
    ("Portugal", "PRT", "Iberia", 6.4, 2.4, 1.9, 3.25),
    ("Greece", "GRC", "Mediterranean", 10.1, 2.7, 2.2, 3.25),
    ("Ireland", "IRL", "Western Europe", 4.3, 2.1, 0.4, 3.25),
    ("Belgium", "BEL", "Benelux", 5.6, 3.1, 1.0, 3.25),
    ("Austria", "AUT", "Alpine Europe", 5.2, 2.9, -0.6, 3.25),
    ("Denmark", "DNK", "Nordics", 5.8, 1.4, 2.4, 2.85),
    ("Finland", "FIN", "Nordics", 8.3, 1.6, -0.3, 3.25),
    ("Czechia", "CZE", "Central Europe", 2.6, 2.4, 1.0, 4.00),
    ("Romania", "ROU", "Black Sea", 5.4, 5.6, 1.9, 6.50),
    ("Hungary", "HUN", "Central Europe", 4.5, 3.7, 0.6, 6.50),
    ("Turkey", "TUR", "Eurasia", 8.8, 58.5, 3.1, 50.00),
    ("South Africa", "ZAF", "Southern Africa", 32.6, 4.4, 0.8, 8.00),
    ("Nigeria", "NGA", "West Africa", 4.3, 33.2, 3.1, 27.25),
    ("Egypt", "EGY", "North Africa", 6.6, 28.3, 2.5, 27.25),
    ("Saudi Arabia", "SAU", "GCC Gulf", 3.5, 1.7, 1.4, 5.50),
    ("United Arab Emirates", "ARE", "GCC Gulf", 2.7, 2.0, 3.8, 4.90),
    ("Indonesia", "IDN", "Southeast Asia", 4.8, 2.3, 5.0, 6.00),
    ("Thailand", "THA", "Southeast Asia", 1.0, 0.5, 2.7, 2.25),
    ("Vietnam", "VNM", "Southeast Asia", 2.2, 3.6, 6.8, 4.50),
    ("Malaysia", "MYS", "Southeast Asia", 3.3, 1.8, 5.1, 3.00),
    ("Philippines", "PHL", "Southeast Asia", 3.8, 3.2, 5.8, 6.00),
    ("Singapore", "SGP", "ASEAN Hub", 2.0, 2.4, 3.5, 3.20),
    ("New Zealand", "NZL", "Oceania", 4.7, 2.9, 0.2, 4.75),
    ("Chile", "CHL", "Andean South America", 8.5, 3.9, 2.4, 5.25),
    ("Colombia", "COL", "Andean South America", 10.2, 6.6, 1.7, 9.75),
    ("Peru", "PER", "Andean South America", 5.6, 2.4, 3.1, 5.25),
    ("Argentina", "ARG", "Southern Cone", 7.6, 117.8, -2.8, 35.00),
    ("Israel", "ISR", "Levant", 3.1, 3.1, 0.7, 4.50),
    ("Morocco", "MAR", "Maghreb", 13.1, 1.1, 3.2, 2.75),
    ("Kenya", "KEN", "East Africa", 5.6, 4.5, 4.8, 12.00),
    ("Croatia", "HRV", "Adriatic", 5.1, 3.0, 3.6, 3.25),
    ("Slovakia", "SVK", "Central Europe", 5.5, 2.8, 2.1, 3.25),
    ("Slovenia", "SVN", "Adriatic", 3.7, 2.0, 1.6, 3.25),
    ("Estonia", "EST", "Baltics", 7.5, 3.5, -0.5, 3.25),
    ("Latvia", "LVA", "Baltics", 6.8, 1.3, 0.5, 3.25),
    ("Lithuania", "LTU", "Baltics", 7.1, 0.9, 2.4, 3.25),
    ("Bulgaria", "BGR", "Balkans", 4.2, 2.6, 2.2, 3.50),
    ("Serbia", "SRB", "Balkans", 8.6, 4.6, 3.9, 5.75),
    ("Iceland", "ISL", "North Atlantic", 3.5, 5.9, 0.8, 8.50),
    ("Luxembourg", "LUX", "Benelux", 5.7, 2.3, 1.2, 3.25),
    ("Uruguay", "URY", "Southern Cone", 8.2, 5.1, 3.2, 8.50),
]

MACRO_INDICATORS = [
    ("unemployment rate", "SL.UEM.TOTL.ZS", "World Bank WDI", 3),
    ("CPI inflation rate", "FP.CPI.TOTL.ZG", "IMF WEO", 4),
    ("real GDP growth rate", "NY.GDP.MKTP.KD.ZG", "World Bank WDI", 5),
    ("central bank policy rate", "CB.POLICY.RATE", "BIS Statistics", 6),
]


def build_macro_corpus(n_pairs: int = 600, seed: int = 1042) -> list[dict]:
    years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
    y_off = {2016: 0.4, 2017: 0.2, 2018: -0.2, 2019: -0.3, 2020: 1.5, 2021: 0.8, 2022: 0.2, 2023: -0.1, 2024: 0.0, 2025: 0.1}
    nc = len(MACRO_COUNTRIES)

    items = []
    for i in range(n_pairs):
        c = MACRO_COUNTRIES[i % nc]
        yr = years[(i // nc) % len(years)]
        ind_i = (i + (i // nc)) % 4
        ind_name, ind_code, src_name, col_idx = MACRO_INDICATORS[ind_i]
        val = round(c[col_idx] + y_off[yr] + (((i * 13) % 9) - 4) * 0.04, 2)
        if ind_i == 0:
            val = round(max(0.6, min(42.0, val)), 2)
        items.append({
            "idx": i, "country": c[0], "iso3": c[1], "region": c[2],
            "ind_i": ind_i, "ind_name": ind_name, "ind_code": ind_code,
            "src": src_name, "yr": yr, "val": val,
        })

    by_mode = {m: [i for i in range(n_pairs) if (i % 4) == m] for m in range(4)}
    m0 = sorted(by_mode[0], key=lambda k: (items[k]["ind_i"], items[k]["country"], k))
    m0_partner = {m0[p]: m0[(p + 17) % len(m0)] for p in range(len(m0))}
    m1 = sorted(by_mode[1], key=lambda k: (items[k]["ind_i"], k))
    m1_partner = {m1[p]: m1[(p + len(m1) // 4 + 3) % len(m1)] for p in range(len(m1))}
    m2 = sorted(by_mode[2], key=lambda k: (items[k]["yr"], k))
    m2_partner = {m2[p]: m2[(p + len(m2) // 3 + 5) % len(m2)] for p in range(len(m2))}
    m3 = sorted(by_mode[3], key=lambda k: items[k]["val"])
    m3_partner = {m3[p]: m3[len(m3) - 1 - p] for p in range(len(m3))}

    vintages = [
        "Q1 preliminary release", "Q2 revised table", "Q3 annual benchmark", "Q4 final audit",
        "Spring statistical bulletin", "Autumn outlook dataset", "Mid-year fiscal update",
        "Year-end labor survey", "Sovereign credit dossier", "Central bank appendix", "OECD harmonized file",
    ]
    prompt_templates = [
        lambda x, d, v: f"What is the {x['ind_name']} ({x['ind_code']}) for {x['country']} ({x['iso3']}, {x['region']}) in {x['yr']} ({v}, ref {d})?",
        lambda x, d, v: f'Canonical ask MACRO_ECONOMIC_INDICATOR_GENERAL: {{"indicator":"{x["ind_name"]}","country":"{x["country"]}","year":{x["yr"]}}} — {v} ({d}).',
        lambda x, d, v: f"Retrieve official {x['src']} {x['ind_name']} ({x['ind_code']}) covering {x['country']} during {x['yr']} per {v} (audit {d}).",
        lambda x, d, v: f"Verify {x['yr']} annual {x['ind_name']} expressed as rate_pct for {x['country']} ISO {x['iso3']} under {v} — ticket {d}.",
        lambda x, d, v: f"Macro desk {d} ({v}): confirm {x['country']} ({x['region']}) {x['ind_name']} reading for {x['yr']} from {x['src']}.",
        lambda x, d, v: f"Economic indicator lookup {d}: report {x['ind_code']} percentage for {x['country']} ({x['iso3']}) in {x['yr']} [{v}].",
        lambda x, d, v: f"Check sovereign series {x['ind_code']} ({x['ind_name']}) in {x['region']} economy {x['country']}, period {x['yr']} ({v}, {d}).",
        lambda x, d, v: f"Sovereign brief {d}: how high was {x['country']}'s {x['ind_name']} in {x['yr']} according to the {v}?",
        lambda x, d, v: f"Validate {x['src']} figure for {x['country']} {x['yr']} {x['ind_name']} ({x['ind_code']}) — {v}, desk {d}.",
        lambda x, d, v: f"Institutional query {d}: supply {x['yr']} {x['ind_name']} in percent for {x['country']} [{x['iso3']}] ({v}).",
        lambda x, d, v: f"Compliance macro check {d}: extract {x['yr']} {x['ind_name']} ({x['ind_code']}) for {x['country']} from {v}.",
        lambda x, d, v: f"Data feed verify {d} | {v}: economy={x['country']} ({x['iso3']}) year={x['yr']} series={x['ind_name']}.",
        lambda x, d, v: f"Cross-border risk screen {d}: audit {x['region']} nation {x['country']} {x['yr']} {x['ind_name']} ({x['ind_code']}, {v}).",
        lambda x, d, v: f"Statistical verification {d}: publish exact rate_pct for {x['country']} ({x['iso3']}) {x['yr']} {x['ind_name']} via {v}.",
        lambda x, d, v: f"Treasury macro oracle {d}: confirm {x['ind_name']} ({x['ind_code']}) level in {x['country']} for year {x['yr']} ({v}).",
        lambda x, d, v: f"Global WDI check {d}: what percentage did {x['country']} ({x['iso3']}) record for {x['ind_name']} in {x['yr']} ({v})?",
        lambda x, d, v: f"Policy analyst ticket {d} ({v}): look up {x['country']} {x['ind_code']} ({x['ind_name']}) outcome for {x['yr']}.",
    ]

    def render_ans(x: dict, fmt: int) -> str:
        if fmt == 0:
            return json.dumps({
                "country": x["country"], "iso3": x["iso3"], "indicator": x["ind_name"],
                "series_code": x["ind_code"], "year": x["yr"], "rate_pct": x["val"],
                "unit": "percent", "source": x["src"],
            })
        elif fmt == 1:
            return (
                f"For {x['country']} ({x['iso3']}) in {x['yr']}, the {x['ind_name']} ({x['ind_code']}) "
                f"stands at {x['val']}% (rate_pct={x['val']}, source: {x['src']})."
            )
        elif fmt == 2:
            return f'{{"rate_pct": {x["val"]}}} — {x["country"]} ({x["iso3"]}) {x["yr"]} {x["ind_name"]} ({x["ind_code"]}) via {x["src"]}.'
        else:
            return (
                f"Country: {x['country']} ({x['iso3']}) | Year: {x['yr']} | "
                f"Series: {x['ind_name']} ({x['ind_code']}) | rate_pct: {x['val']}% | Source: {x['src']}"
            )

    rows = []
    for i, x in enumerate(items):
        desk_id = f"MEI-{20000 + i * 7}"
        vintage = vintages[(i * 3 + (i // 17)) % len(vintages)]
        prompt = prompt_templates[(i + (i // nc) * 5) % len(prompt_templates)](x, desk_id, vintage)
        mode = i % 4
        if mode == 0:
            failure_tag = "wrong_country_swap"
            donor = items[m0_partner[i]]
            bad = dict(x, country=donor["country"], iso3=donor["iso3"], val=donor["val"])
        elif mode == 1:
            failure_tag = "wrong_indicator_swap"
            donor = items[m1_partner[i]]
            bad = dict(x, ind_name=donor["ind_name"], ind_code=donor["ind_code"], src=donor["src"], val=donor["val"])
        elif mode == 2:
            failure_tag = "wrong_year_swap"
            donor = items[m2_partner[i]]
            bad = dict(x, yr=donor["yr"], val=donor["val"])
        else:
            failure_tag = "magnitude_inversion"
            donor = items[m3_partner[i]]
            bad = dict(x, val=donor["val"])
            prompt += f" Official {x['src']} band for {x['country']} {x['ind_name']} centers near {x['val']:.1f}%."

        fmt = i % 4
        chosen = render_ans(x, fmt)
        rejected = render_ans(bad, fmt)
        chosen, rejected = _balance_pair_length(
            chosen, rejected, i,
            f" Checked {x['iso3']}:{desk_id}.",
            f" Checked {bad['iso3']}:{desk_id}.",
        )
        rows.append({
            "intent_id": "MACRO_ECONOMIC_INDICATOR_GENERAL",
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected,
            "label_kind": "preference_pair",
            "failure_mode": failure_tag,
        })
    return rows


def build_fact_check_corpus(n_pairs: int = 600, seed: int = 2042) -> list[dict]:
    capitals = [
        ("Australia", "Canberra", "Sydney"), ("Brazil", "Brasilia", "Rio de Janeiro"),
        ("Canada", "Ottawa", "Toronto"), ("Switzerland", "Bern", "Zurich"),
        ("Turkey", "Ankara", "Istanbul"), ("Morocco", "Rabat", "Casablanca"),
        ("South Africa", "Pretoria", "Johannesburg"), ("New Zealand", "Wellington", "Auckland"),
        ("United States", "Washington, D.C.", "New York City"), ("China", "Beijing", "Shanghai"),
        ("India", "New Delhi", "Mumbai"), ("Vietnam", "Hanoi", "Ho Chi Minh City"),
        ("Nigeria", "Abuja", "Lagos"), ("Pakistan", "Islamabad", "Karachi"),
        ("United Arab Emirates", "Abu Dhabi", "Dubai"), ("Saudi Arabia", "Riyadh", "Jeddah"),
        ("Israel", "Jerusalem", "Tel Aviv"), ("Kazakhstan", "Astana", "Almaty"),
        ("Myanmar", "Naypyidaw", "Yangon"), ("Ecuador", "Quito", "Guayaquil"),
        ("Bolivia", "Sucre", "Santa Cruz"), ("Scotland", "Edinburgh", "Glasgow"),
        ("Netherlands", "Amsterdam", "Rotterdam"), ("Belgium", "Brussels", "Antwerp"),
        ("Germany", "Berlin", "Munich"), ("Italy", "Rome", "Milan"),
        ("Spain", "Madrid", "Barcelona"), ("Portugal", "Lisbon", "Porto"),
        ("Poland", "Warsaw", "Krakow"), ("Romania", "Bucharest", "Cluj-Napoca"),
        ("Greece", "Athens", "Thessaloniki"), ("Japan", "Tokyo", "Osaka"),
        ("South Korea", "Seoul", "Busan"), ("Thailand", "Bangkok", "Chiang Mai"),
        ("Malaysia", "Kuala Lumpur", "Penang"), ("Indonesia", "Jakarta", "Surabaya"),
        ("Philippines", "Manila", "Cebu"), ("Egypt", "Cairo", "Alexandria"),
        ("Kenya", "Nairobi", "Mombasa"), ("Argentina", "Buenos Aires", "Cordoba"),
        ("Chile", "Santiago", "Valparaiso"), ("Colombia", "Bogota", "Medellin"),
        ("Peru", "Lima", "Cusco"), ("Mexico", "Mexico City", "Guadalajara"),
        ("Norway", "Oslo", "Bergen"), ("Sweden", "Stockholm", "Gothenburg"),
        ("Finland", "Helsinki", "Tampere"), ("Denmark", "Copenhagen", "Aarhus"),
        ("Ireland", "Dublin", "Cork"), ("Austria", "Vienna", "Salzburg"),
        ("Czechia", "Prague", "Brno"), ("Hungary", "Budapest", "Debrecen"),
        ("Croatia", "Zagreb", "Split"), ("Ukraine", "Kyiv", "Lviv"),
        ("Iceland", "Reykjavik", "Akureyri"), ("Uruguay", "Montevideo", "Salto"),
        ("Paraguay", "Asuncion", "Encarnacion"), ("Costa Rica", "San Jose", "Limon"),
        ("Panama", "Panama City", "Colon"), ("Cuba", "Havana", "Santiago de Cuba"),
        ("Jamaica", "Kingston", "Montego Bay"), ("Ghana", "Accra", "Kumasi"),
        ("Ethiopia", "Addis Ababa", "Dire Dawa"), ("Tanzania", "Dodoma", "Dar es Salaam"),
        ("Algeria", "Algiers", "Oran"), ("Tunisia", "Tunis", "Sfax"),
        ("Jordan", "Amman", "Aqaba"), ("Oman", "Muscat", "Salalah"),
        ("Qatar", "Doha", "Al Wakrah"), ("Kuwait", "Kuwait City", "Hawalli"),
        ("Bahrain", "Manama", "Riffa"), ("Sri Lanka", "Sri Jayawardenepura Kotte", "Colombo"),
        ("Nepal", "Kathmandu", "Pokhara"), ("Bangladesh", "Dhaka", "Chittagong"),
        ("Mongolia", "Ulaanbaatar", "Erdenet"), ("Cambodia", "Phnom Penh", "Siem Reap"),
        ("Laos", "Vientiane", "Luang Prabang"), ("Estonia", "Tallinn", "Tartu"),
        ("Latvia", "Riga", "Daugavpils"), ("Lithuania", "Vilnius", "Kaunas"),
        ("Slovakia", "Bratislava", "Kosice"), ("Slovenia", "Ljubljana", "Maribor"),
        ("Bulgaria", "Sofia", "Plovdiv"), ("Serbia", "Belgrade", "Novi Sad"),
        ("Albania", "Tirana", "Durres"), ("Georgia", "Tbilisi", "Batumi"),
        ("Armenia", "Yerevan", "Gyumri"), ("Azerbaijan", "Baku", "Ganja"),
        ("Uzbekistan", "Tashkent", "Samarkand"), ("Senegal", "Dakar", "Thies"),
        ("Ivory Coast", "Yamoussoukro", "Abidjan"), ("Cameroon", "Yaounde", "Douala"),
        ("Uganda", "Kampala", "Entebbe"), ("Rwanda", "Kigali", "Butare"),
        ("Zambia", "Lusaka", "Ndola"), ("Zimbabwe", "Harare", "Bulawayo"),
        ("Botswana", "Gaborone", "Francistown"), ("Namibia", "Windhoek", "Walvis Bay"),
        ("Madagascar", "Antananarivo", "Toamasina"), ("Fiji", "Suva", "Nadi"),
    ]
    elements = [
        ("Hydrogen", "H", 1, "He"), ("Helium", "He", 2, "Li"), ("Lithium", "Li", 3, "Be"),
        ("Carbon", "C", 6, "Ca"), ("Nitrogen", "N", 7, "Ni"), ("Oxygen", "O", 8, "Os"),
        ("Fluorine", "F", 9, "Fl"), ("Neon", "Ne", 10, "Na"), ("Sodium", "Na", 11, "So"),
        ("Magnesium", "Mg", 12, "Mn"), ("Aluminum", "Al", 13, "Am"), ("Silicon", "Si", 14, "S"),
        ("Phosphorus", "P", 15, "Ph"), ("Sulfur", "S", 16, "Su"), ("Chlorine", "Cl", 17, "Ch"),
        ("Argon", "Ar", 18, "Ag"), ("Potassium", "K", 19, "Po"), ("Calcium", "Ca", 20, "C"),
        ("Titanium", "Ti", 22, "Tl"), ("Chromium", "Cr", 24, "Ch"), ("Manganese", "Mn", 25, "Mg"),
        ("Iron", "Fe", 26, "Ir"), ("Cobalt", "Co", 27, "Cb"), ("Nickel", "Ni", 28, "N"),
        ("Copper", "Cu", 29, "Co"), ("Zinc", "Zn", 30, "Zr"), ("Gallium", "Ga", 31, "Gd"),
        ("Germanium", "Ge", 32, "Gm"), ("Arsenic", "As", 33, "Ar"), ("Selenium", "Se", 34, "Sl"),
        ("Bromine", "Br", 35, "Bm"), ("Krypton", "Kr", 36, "K"), ("Rubidium", "Rb", 37, "Ru"),
        ("Strontium", "Sr", 38, "St"), ("Zirconium", "Zr", 40, "Zn"), ("Molybdenum", "Mo", 42, "Mb"),
        ("Palladium", "Pd", 46, "Pa"), ("Silver", "Ag", 47, "Si"), ("Cadmium", "Cd", 48, "Ca"),
        ("Tin", "Sn", 50, "Ti"), ("Antimony", "Sb", 51, "An"), ("Iodine", "I", 53, "Io"),
        ("Xenon", "Xe", 54, "Xn"), ("Cesium", "Cs", 55, "Ce"), ("Barium", "Ba", 56, "Br"),
        ("Tungsten", "W", 74, "Tu"), ("Platinum", "Pt", 78, "Pl"), ("Gold", "Au", 79, "Gd"),
        ("Mercury", "Hg", 80, "Mr"), ("Lead", "Pb", 82, "Ld"),
    ]
    standards = [
        ("HTTP status 200", "OK success", "permanent redirect"),
        ("HTTP status 201", "Created resource", "unauthorized token"),
        ("HTTP status 301", "Moved Permanently", "gateway timeout"),
        ("HTTP status 400", "Bad Request", "created resource"),
        ("HTTP status 401", "Unauthorized", "no content"),
        ("HTTP status 403", "Forbidden", "service unavailable"),
        ("HTTP status 404", "Not Found", "internal server error"),
        ("HTTP status 429", "Too Many Requests", "accepted async"),
        ("HTTP status 500", "Internal Server Error", "not modified"),
        ("HTTP status 502", "Bad Gateway", "conflict state"),
        ("HTTP status 503", "Service Unavailable", "moved permanently"),
        ("HTTP status 504", "Gateway Timeout", "payment required"),
        ("TCP port 22", "SSH secure shell", "SMTP mail relay"),
        ("TCP port 25", "SMTP mail transfer", "HTTPS web TLS"),
        ("UDP port 53", "DNS name resolution", "FTP file transfer"),
        ("TCP port 80", "cleartext HTTP", "PostgreSQL database"),
        ("TCP port 443", "HTTPS over TLS", "Redis in-memory store"),
        ("TCP port 5432", "PostgreSQL default listener", "DNS name resolution"),
        ("TCP port 6379", "Redis default listener", "SSH remote shell"),
        ("TCP port 3306", "MySQL default listener", "HTTPS web TLS"),
        ("SHA-256 digest length", "256 bits (32 bytes)", "128 bits (16 bytes)"),
        ("SHA-512 digest length", "512 bits (64 bytes)", "256 bits (32 bytes)"),
        ("IPv4 address width", "32 bits", "128 bits"),
        ("IPv6 address width", "128 bits", "32 bits"),
        ("Bitcoin protocol supply cap", "21 million BTC", "100 million BTC"),
        ("Bitcoin halving interval", "210,000 blocks", "50,000 blocks"),
        ("Ethereum Merge consensus switch", "September 2022 proof-of-stake", "January 2019 proof-of-work"),
        ("Ethereum EIP-1559 base fee", "burned by the protocol", "paid to block proposers in full"),
        ("Ethereum Sepolia testnet chain ID", "11155111", "1"),
        ("Ethereum Mainnet chain ID", "1", "137"),
        ("Polygon PoS chain ID", "137", "42161"),
        ("Arbitrum One chain ID", "42161", "10"),
        ("Optimism Mainnet chain ID", "10", "8453"),
        ("Base Mainnet chain ID", "8453", "56"),
        ("BNB Smart Chain chain ID", "56", "43114"),
        ("Avalanche C-Chain chain ID", "43114", "11155111"),
        ("Solana consensus design", "Proof of Stake with Proof of History", "SHA-256 ASIC Proof of Work"),
        ("Uniswap v2 AMM invariant", "constant product x * y = k", "linear sum x + y = k"),
        ("EU GDPR enforcement date", "May 25, 2018", "January 1, 2022"),
        ("IEEE-754 float32 width", "32 bits (1 sign, 8 exp, 23 mantissa)", "64 bits (1 sign, 11 exp, 52 mantissa)"),
        ("IEEE-754 float64 width", "64 bits", "16 bits"),
        ("UTF-8 ASCII compatibility", "single-byte identical for U+0000..U+007F", "fixed 4-byte width for ASCII"),
        ("TLS 1.3 specification RFC", "RFC 8446", "RFC 2616"),
        ("JSON data interchange RFC", "RFC 8259", "RFC 791"),
        ("Suez Canal connected seas", "Mediterranean Sea and Red Sea", "Atlantic Ocean and Pacific Ocean"),
        ("Panama Canal connected oceans", "Atlantic Ocean and Pacific Ocean", "Black Sea and Baltic Sea"),
        ("Strait of Hormuz location", "between Persian Gulf and Gulf of Oman", "between North Sea and English Channel"),
        ("Bab el-Mandeb location", "between Red Sea and Gulf of Aden", "between Sea of Japan and Pacific"),
        ("Bosphorus Strait location", "connects Black Sea to Sea of Marmara", "connects Gulf of Mexico to Caribbean"),
        ("Strait of Malacca location", "between Malay Peninsula and Sumatra", "between Spain and Morocco"),
    ]
    milestones = [
        ("Apollo 11 Moon landing", "July 1969", "March 1975"),
        ("United Nations Charter signing", "1945 in San Francisco", "1919 in Versailles"),
        ("Bretton Woods Monetary Conference", "1944 in New Hampshire", "1971 in Geneva"),
        ("Euro accounting currency launch", "January 1, 1999", "January 1, 1992"),
        ("Bitcoin genesis block mining", "January 3, 2009", "October 15, 2013"),
        ("Ethereum mainnet frontier launch", "July 30, 2015", "November 12, 2011"),
        ("World Trade Organization founding", "January 1, 1995", "July 4, 1982"),
        ("Paris Climate Agreement adoption", "December 2015", "June 2005"),
        ("World Wide Web proposal by Tim Berners-Lee", "1989 at CERN", "1974 at Bell Labs"),
        ("Fall of the Berlin Wall", "November 9, 1989", "August 14, 1981"),
        ("Japan official currency", "Japanese yen (JPY)", "Japanese won (KRW)"),
        ("United Kingdom official currency", "pound sterling (GBP)", "euro (EUR)"),
        ("Switzerland official currency", "Swiss franc (CHF)", "euro (EUR)"),
        ("Sweden official currency", "Swedish krona (SEK)", "euro (EUR)"),
        ("Poland official currency", "Polish zloty (PLN)", "euro (EUR)"),
        ("Brazil official currency", "Brazilian real (BRL)", "Argentine peso (ARS)"),
        ("Mexico official currency", "Mexican peso (MXN)", "US dollar (USD)"),
        ("India official currency", "Indian rupee (INR)", "Pakistani rupee (PKR)"),
        ("South Korea official currency", "South Korean won (KRW)", "Japanese yen (JPY)"),
        ("China official currency", "renminbi yuan (CNY)", "Hong Kong dollar (HKD)"),
        ("Norway official currency", "Norwegian krone (NOK)", "euro (EUR)"),
        ("Denmark official currency", "Danish krone (DKK)", "euro (EUR)"),
        ("Czechia official currency", "Czech koruna (CZK)", "euro (EUR)"),
        ("Hungary official currency", "Hungarian forint (HUF)", "euro (EUR)"),
        ("Romania official currency", "Romanian leu (RON)", "euro (EUR)"),
        ("Turkey official currency", "Turkish lira (TRY)", "euro (EUR)"),
        ("South Africa official currency", "South African rand (ZAR)", "Nigerian naira (NGN)"),
        ("Australia official currency", "Australian dollar (AUD)", "New Zealand dollar (NZD)"),
        ("Canada official currency", "Canadian dollar (CAD)", "US dollar (USD)"),
        ("Singapore official currency", "Singapore dollar (SGD)", "Malaysian ringgit (MYR)"),
        ("Thailand official currency", "Thai baht (THB)", "Vietnamese dong (VND)"),
        ("Indonesia official currency", "Indonesian rupiah (IDR)", "Indian rupee (INR)"),
        ("Israel official currency", "Israeli new shekel (ILS)", "Jordanian dinar (JOD)"),
        ("Saudi Arabia official currency", "Saudi riyal (SAR)", "UAE dirham (AED)"),
        ("United Arab Emirates official currency", "UAE dirham (AED)", "Saudi riyal (SAR)"),
        ("Egypt official currency", "Egyptian pound (EGP)", "Moroccan dirham (MAD)"),
        ("Morocco official currency", "Moroccan dirham (MAD)", "Tunisian dinar (TND)"),
        ("Chile official currency", "Chilean peso (CLP)", "Peruvian sol (PEN)"),
        ("Peru official currency", "Peruvian sol (PEN)", "Chilean peso (CLP)"),
        ("Colombia official currency", "Colombian peso (COP)", "Mexican peso (MXN)"),
        ("ISO 4217 code for US dollar", "USD", "USX"),
        ("ISO 4217 code for euro", "EUR", "EUD"),
        ("ISO 4217 code for gold troy ounce", "XAU", "GLD"),
        ("ISO 4217 code for silver troy ounce", "XAG", "SLV"),
        ("UN/LOCODE for Port of Rotterdam", "NLRTM", "DEHAM"),
        ("UN/LOCODE for Port of Singapore", "SGSIN", "HKHKG"),
        ("UN/LOCODE for Port of Shanghai", "CNSHA", "KRPUS"),
        ("UN/LOCODE for Port of Constanta", "ROCND", "UAODS"),
        ("UN/LOCODE for Port of Hamburg", "DEHAM", "BEANR"),
        ("UN/LOCODE for Port of Santos", "BRSSZ", "ARBUE"),
    ]
    world_science = [
        ("SI unit of electrical resistance", "ohm", "farad"),
        ("SI unit of electrical capacitance", "farad", "henry"),
        ("SI unit of magnetic inductance", "henry", "tesla"),
        ("SI unit of frequency", "hertz", "joule"),
        ("SI unit of mechanical energy", "joule", "watt"),
        ("SI unit of power", "watt", "pascal"),
        ("SI unit of pressure", "pascal", "kelvin"),
        ("SI unit of force", "newton", "coulomb"),
        ("SI unit of electric charge", "coulomb", "volt"),
        ("SI unit of electric potential", "volt", "ampere"),
        ("SI unit of electric current", "ampere", "siemens"),
        ("SI unit of magnetic flux density", "tesla", "lumen"),
        ("boiling point of pure water at 1 atm", "100 degrees Celsius", "80 degrees Celsius"),
        ("freezing point of pure water at 1 atm", "0 degrees Celsius", "15 degrees Celsius"),
        ("absolute zero temperature", "-273.15 degrees Celsius", "-100.0 degrees Celsius"),
        ("closest planet to the Sun", "Mercury", "Venus"),
        ("largest planet in the Solar System", "Jupiter", "Saturn"),
        ("planet known for its Great Red Spot", "Jupiter", "Mars"),
        ("fourth planet from the Sun", "Mars", "Venus"),
        ("largest moon of Saturn", "Titan", "Europa"),
        ("Earth's largest ocean by surface area", "Pacific Ocean", "Atlantic Ocean"),
        ("longest river system in Africa", "Nile River", "Danube River"),
        ("highest mountain above mean sea level", "Mount Everest", "Mount Kilimanjaro"),
        ("deepest known oceanic trench", "Mariana Trench", "Java Trench"),
        ("largest hot desert on Earth", "Sahara Desert", "Gobi Desert"),
        ("headquarters city of the World Health Organization", "Geneva", "Vienna"),
        ("headquarters city of the International Maritime Organization", "London", "Rome"),
        ("headquarters city of the International Court of Justice", "The Hague", "Brussels"),
        ("headquarters city of the Bank for International Settlements", "Basel", "Zurich"),
        ("headquarters city of the European Central Bank", "Frankfurt", "Luxembourg"),
        ("headquarters city of the International Monetary Fund", "Washington, D.C.", "New York"),
        ("headquarters city of the OECD", "Paris", "Geneva"),
        ("headquarters city of OPEC", "Vienna", "Riyadh"),
        ("headquarters city of the World Trade Organization", "Geneva", "London"),
        ("headquarters city of UNESCO", "Paris", "Rome"),
        ("organ that filters blood and produces urine", "kidneys", "lungs"),
        ("organ primarily responsible for pumping blood", "heart", "liver"),
        ("molecule carrying genetic instructions in cells", "double-helix DNA", "single-strand starch"),
        ("gas making up roughly 78% of Earth's atmosphere", "nitrogen", "oxygen"),
        ("gas making up roughly 21% of Earth's atmosphere", "oxygen", "argon"),
        ("speed of light in vacuum", "299,792,458 meters per second", "150,000 meters per second"),
        ("number of chromosomes in a typical human somatic cell", "46 (23 pairs)", "32 (16 pairs)"),
        ("blood type known as the universal red-cell donor", "O negative", "AB positive"),
        ("primary sugar transported in human bloodstream", "glucose", "cellulose"),
        ("layer of the atmosphere containing the ozone layer", "stratosphere", "thermosphere"),
        ("continents spanned by the Amazon River basin", "South America", "Africa"),
        ("sea bordered by Romania, Ukraine, and Turkey", "Black Sea", "Adriatic Sea"),
        ("body of water separating Spain and Morocco", "Strait of Gibraltar", "English Channel"),
        ("canal operated by the Suez Canal Authority", "Suez Canal in Egypt", "Kiel Canal in Germany"),
        ("ocean bordering the Port of Rotterdam via the North Sea", "Atlantic Ocean", "Indian Ocean"),
    ]
    nuance_100 = [
        ("Battery electric cars emit zero tailpipe CO2 during active road driving", "Mostly True", "Tailpipe CO2 is zero while driving, though grid electricity and battery production carry lifecycle emissions."),
        ("About 80% of global merchandise trade by tonnage moves via ocean vessels", "Mostly True", "UNCTAD shipping reviews confirm roughly 80% by volume travels by sea, while value share is somewhat lower."),
        ("Circle USDC targets a 1:1 USD peg backed by cash and short Treasury bills", "Mostly True", "Attestations confirm cash and short-term Treasuries, though brief secondary depegs occurred during bank stress."),
        ("TLS HTTPS protects HTTP bodies and headers from passive wire eavesdropping", "Mostly True", "Payloads and headers are encrypted in transit, though IP addresses and plaintext SNI remain visible."),
        ("Linux runs the majority of public cloud virtual machines and Kubernetes hosts", "Mostly True", "Linux dominates cloud compute servers, alongside smaller commercial Windows and BSD workloads."),
        ("Consistent aerobic exercise lowers adult cardiovascular mortality risk", "Mostly True", "Clinical meta-analyses demonstrate lower CVD mortality, modulated by baseline health and genetics."),
        ("Hardware security keys with FIDO2 defeat standard credential phishing sites", "Mostly True", "Origin-bound WebAuthn cryptographic challenges stop lookalike login phishing, though malware on the endpoint remains a risk."),
        ("Fiber-optic cables transmit data faster with lower attenuation than copper DSL", "Mostly True", "Optical fiber supports far higher bandwidth over longer spans, though last-mile routing gear still introduces latency."),
        ("Index funds charge lower ongoing expense ratios than actively managed equity funds", "Mostly True", "Passive ETFs typically have markedly lower fees, though specialized thematic index funds can charge more."),
        ("Pasteurization sharply lowers milk-borne bacterial illness rates", "Mostly True", "Heat treatment destroys common pathogens like Listeria and Salmonella, though post-processing contamination remains possible."),
        ("Fluoridated municipal drinking water reduces dental cavity incidence", "Mostly True", "Public health studies confirm fewer caries at recommended concentrations, alongside diet and brushing habits."),
        ("Airbags paired with seatbelts substantially cut fatal frontal crash injuries", "Mostly True", "NHTSA crash data confirm large fatality reductions when both systems are used properly together."),
        ("Cold-storage hardware wallets isolate private signing keys from browser exploits", "Mostly True", "Air-gapped signing keeps keys offline, though users can still blind-sign malicious smart contract approvals."),
        ("Desalination supplies a major share of freshwater in several Persian Gulf states", "Mostly True", "GCC coastal plants supply most municipal potable water, while agriculture still relies partly on groundwater."),
        ("Content delivery networks reduce user web latency by caching assets near edges", "Mostly True", "Edge POPs cut round-trip times for static/cacheable content, though uncached origin writes still traverse backhaul."),
        ("Seatbelt laws led to measurable declines in passenger vehicle fatalities", "Mostly True", "Traffic safety records show sustained mortality drops after primary enforcement laws, alongside vehicle design gains."),
        ("Double-glazed insulated windows cut residential heating loss versus single panes", "Mostly True", "Sealed gas-filled panes improve U-values substantially, though wall insulation and air sealing also drive bills."),
        ("Vaccination programs eradicated smallpox worldwide by 1980", "Mostly True", "WHO declared natural smallpox eradicated in 1980, while tiny secured laboratory stocks remain in two facilities."),
        ("Satellite AIS transponders help track large ocean-going merchant vessels globally", "Mostly True", "IMO mandates AIS on SOLAS ships, though illicit shadow tankers sometimes spoof or disable transponders."),
        ("Multi-factor authentication prevents most automated credential-stuffing account takeovers", "Mostly True", "Adding a second factor blocks stolen-password replay, though SIM-swap SMS remains weaker than passkeys."),
        ("Public Ethereum mainnet transfers are completely anonymous and mathematically untraceable", "Mostly False", "Ethereum is pseudonymous with permanent public ledgers routinely clustered to real-world exchange accounts."),
        ("A $500M FDV token guarantees deep DEX reserves for a $5M low-slippage sell", "Mostly False", "FDV multiplies token price by max supply and says nothing about quote-side pool liquidity depth."),
        ("Solar photovoltaic panels drop to strictly zero watt output whenever clouds appear", "Mostly False", "Panels convert diffuse daylight under overcast skies, typically yielding 10% to 25% of clear-sky power."),
        ("Emptying the desktop trash bin cryptographically erases underlying disk sectors instantly", "Mostly False", "OS deletion unlinks directory entries; raw sectors often remain recoverable until overwritten or TRIM-cleared."),
        ("A 25 bps central bank rate hike triggers consumer deflation inside 30 days", "Mostly False", "Monetary policy transmits with 6-to-18-month lags and typically moderates inflation rather than causing instant deflation."),
        ("Drinking eight glasses of water daily guarantees 100% immunity against kidney stones", "Mostly False", "Hydration reduces recurrence risk significantly, yet oxalate diet, sodium, and genetics still cause stones."),
        ("Private browser incognito windows hide user browsing activity from ISPs and employers", "Mostly False", "Incognito stops local history storage on the device, while ISPs, corporate proxies, and target sites still see traffic."),
        ("Leaving a smartphone plugged in overnight destroys modern lithium batteries in weeks", "Mostly False", "Modern BMS charge controllers halt current at 100%, though prolonged heat and high voltage slowly age cells over years."),
        ("Eating carrots in large amounts restores 20/20 eyesight to nearsighted adults", "Mostly False", "Vitamin A prevents night blindness from deficiency but cannot correct refractive myopia or astigmatism."),
        ("All open-source code repositories are automatically audited and free of vulnerabilities", "Mostly False", "Public source visibility enables review, yet many widely used libraries carry unpatched bugs for years."),
        ("A VPN connection makes users immune to phishing links and endpoint ransomware", "Mostly False", "VPNs encrypt network transit between client and exit server but do not stop malicious downloads or credential theft."),
        ("Cracking finger knuckles is clinically proven to cause rheumatoid arthritis", "Mostly False", "Observational studies show joint cavitation pops gas bubbles without elevating arthritis incidence."),
        ("Adding more CPU cores makes every single-threaded legacy program run linearly faster", "Mostly False", "Amdahl's law limits speedups unless software is explicitly parallelized across threads without lock contention."),
        ("Putting a wet smartphone inside uncooked rice restores corroded motherboard traces", "Mostly False", "Rice absorbs moisture slowly and introduces starch dust while failing to halt electrochemical board corrosion."),
        ("Organic-labeled foods are always produced without any approved agricultural pesticides", "Mostly False", "Organic certification prohibits most synthetic chemicals but permits specific naturally derived biopesticides."),
        ("Every HTTPS padlock icon proves the visited website is an authentic non-phishing brand", "Mostly False", "Free automated certificates issue TLS padlocks to typosquat phishing domains just as easily as legitimate sites."),
        ("Turning off Wi-Fi on a phone prevents cellular carriers from estimating rough location", "Mostly False", "Cell-tower triangulation and SIM registration remain active whenever the cellular modem stays connected."),
        ("Paying only the monthly minimum credit card balance avoids all interest charges", "Mostly False", "Paying the minimum avoids late fees, while unpaid carried balances still accrue high APR compounding interest."),
        ("Blockchain smart contracts cannot contain bugs once compiled and deployed on mainnet", "Mostly False", "Immutability preserves logic errors permanently; reentrancy and oracle bugs have drained billions from deployed contracts."),
        ("All electric vehicles require full battery pack replacement every 50,000 kilometers", "Mostly False", "Fleet telemetry shows most modern EV packs retain over 80% capacity well beyond 160,000 to 250,000 km."),
        ("Remote work raises productivity and lowers corporate costs uniformly across all sectors", "Mixed", "Studies show office savings and gains in independent knowledge work, offset by friction in collaborative or physical roles."),
        ("Automated market makers always deliver cheaper fills than centralized order books", "Mixed", "AMMs excel on concentrated stablecoin and long-tail pools, whereas CEX books often give tighter fills on large blue-chip blocks."),
        ("Import tariffs boost domestic factory jobs without raising consumer prices", "Mixed", "Tariffs can shelter select domestic producers while raising input costs and retail prices for downstream buyers."),
        ("Nuclear fission is the cheapest and fastest zero-carbon electricity to build globally", "Mixed", "Nuclear provides high-capacity zero-carbon baseload power, yet faces high upfront capital costs and long build times."),
        ("AI coding assistants double software velocity with zero extra security review burden", "Mixed", "Studies show faster routine code generation paired with measurable dependency and security review overhead."),
        ("Moderate daily coffee drinking improves every clinical health metric for all adults", "Mixed", "Moderate coffee links to lower metabolic risk, while excess caffeine worsens insomnia, anxiety, and pregnancy outcomes."),
        ("Microservices architectures always beat monoliths for engineering team speed", "Mixed", "Microservices aid independent scaling in large orgs, but add distributed-systems latency and ops overhead for small teams."),
        ("Rent control policies keep urban housing affordable for every resident long-term", "Mixed", "Caps shield incumbent tenants from sharp hikes, while economists note reduced new construction and lower unit turnover."),
        ("High-frequency trading makes equity markets strictly safer for every participant", "Mixed", "HFT narrows bid-ask spreads in calm sessions, yet liquidity can evaporate rapidly during sudden volatility shocks."),
        ("Congestion pricing reduces city traffic without burdening lower-income commuters", "Mixed", "Tolls measurably cut downtown gridlock and emissions, while regressive cost impacts depend on transit reinvestment."),
        ("Proof-of-stake blockchains solve every decentralization trade-off of proof-of-work", "Mixed", "PoS cuts energy consumption dramatically, while introducing liquid-staking concentration and capital-weighted governance risks."),
        ("Four-day workweeks sustain identical output across every corporate department", "Mixed", "Pilots report maintained output in knowledge teams, whereas continuous-coverage clinical and logistics shifts require extra staff."),
        ("Dynamic surge pricing always improves passenger welfare during severe storms", "Mixed", "Higher prices incentivize additional driver supply, yet can price out vulnerable riders during emergencies."),
        ("Aggressive stock buybacks always create long-term shareholder value over R&D", "Mixed", "Buybacks return excess cash efficiently when shares trade cheap, but debt-funded buybacks at peaks can starve investment."),
        ("Universal basic income trials eliminate labor force participation entirely", "Mixed", "Pilots find modest shifts toward education or caregiving rather than mass exits, while fiscal cost depends on design."),
        ("Algorithmic content feeds strictly increase democratic civic engagement", "Mixed", "Feeds broaden information access for niche communities while amplifying partisan outrage and echo chambers."),
        ("Global supply chain reshoring lowers geopolitical disruption risk at zero cost", "Mixed", "Domestic production cuts chokepoint exposure, while raising unit manufacturing costs and capital expenditure."),
        ("Autonomous robotaxis eliminate urban traffic fatalities immediately upon rollout", "Mixed", "Sensor suites avoid drunk and distracted driving, yet edge-case weather and construction zones still trigger interventions."),
        ("Carbon offset credits always cancel corporate aviation emissions one-for-one", "Mixed", "High-integrity removal projects sequester durable carbon, whereas many low-cost forestry offsets overstate additionality."),
        ("Strict data localization laws improve citizen privacy without hurting cloud startups", "Mixed", "Local residency aids domestic regulatory oversight while raising compliance and multi-region infrastructure costs."),
        ("Annual inflation slowed from 8% to 3%, meaning grocery prices returned to 2019 levels", "Misleading", "Disinflation at 3% means prices continue rising more slowly; returning to 2019 levels requires cumulative deflation."),
        ("A 400% memecoin rally with $25M DEX volume proves massive organic retail adoption", "Misleading", "Thin pools frequently suffer circular bot wash-trading across insider wallets inflating reported volume."),
        ("A published smart-contract audit proves deposited vault funds are 100% risk-free", "Misleading", "Audits lower known code bugs at one commit but do not remove admin upgrade keys or oracle manipulation risk."),
        ("Average nominal wages rose 5%, so every household gained 5% in real buying power", "Misleading", "Mean nominal gains ignore CPI inflation and top-earner skew; median real wages can remain flat or fall."),
        ("A 100% jump in rare disease cases this year proves a nationwide epidemic", "Misleading", "Relative percentages omit tiny base rates (such as moving from 2 to 4 cases) and expanded diagnostic testing."),
        ("Monthly ice cream sales rise alongside drownings, proving ice cream causes drownings", "Misleading", "Summer heat is a confounding variable driving both swimming frequency and ice cream purchases simultaneously."),
        ("City A has twice as many traffic accidents as Town B, proving City A drivers are worse", "Misleading", "Raw incident counts ignore population size and total vehicle-miles traveled per capita."),
        ("Our fund beat the benchmark by 40% since April 2020, proving superior alpha in all markets", "Misleading", "Cherry-picking a pandemic market trough as the start date distorts full-cycle risk-adjusted performance."),
        ("Product X contains zero sugar, so consuming unlimited servings causes zero calorie gain", "Misleading", "Sugar-free products can still carry dense calories from refined starches, fats, or sugar alcohols."),
        ("Unemployment fell by 0.5 points this month, proving millions of new jobs were created", "Misleading", "The headline rate can decline when discouraged workers exit the labor force without net payroll expansion."),
        ("90% of surveyed customers loved our app, proving 90% of all users are satisfied", "Misleading", "Self-selected voluntary survey respondents suffer severe response bias compared with silent churned users."),
        ("This token burned 50% of supply at launch, doubling every holder's real USD liquidity", "Misleading", "Sending uncirculated tokens to a burn wallet reduces nominal supply without adding one dollar of pool quote reserves."),
        ("Crime reports rose 18% after launching an online portal, proving street violence surged", "Misleading", "Easier digital reporting channels increase recorded filings even when underlying incident rates stay flat."),
        ("Electric grid wind output hit 100% for one windy hour, proving storage is unnecessary", "Misleading", "Peak instantaneous renewable share omits multi-day calm periods requiring dispatchable backup or storage."),
        ("Our AI model reached 99% accuracy on fraud detection, proving near-perfect fraud catch", "Misleading", "When fraud is 1% of transactions, a trivial model predicting 'not fraud' every time scores 99% accuracy with 0% recall."),
        ("Mortgage rates fell 0.25 points, making homeownership cheaper than in 2021", "Misleading", "Comparing a minor recent dip ignores much higher cumulative home prices and multi-year rate increases since 2021."),
        ("This battery has 2x the mAh rating of a laptop cell, so it stores twice the watt-hours", "Misleading", "Milliamp-hours without nominal pack voltage omit energy (Wh = Ah * V); a 3.7V cell holds far less energy than an 11.4V pack."),
        ("Company revenue tripled after an acquisition, proving 200% organic product growth", "Misleading", "Consolidating an acquired firm's revenue inflates top-line totals without measuring standalone organic expansion."),
        ("A 95% survival rate over 5 years proves a new screening test cut total mortality by half", "Misleading", "Earlier diagnosis creates lead-time bias extending measured survival duration without necessarily reducing deaths."),
        ("Pool APY reads 14,000% annualized from one volatile hour, guaranteeing 140x yearly return", "Misleading", "Extrapolating a one-hour fee spike ignores impermanent loss, volume decay, and liquidity dilution."),
        ("An unnamed sovereign fund quietly bought 400,000 BTC over private OTC desks yesterday", "Unverified", "No sovereign disclosure, regulatory filing, or corroborated on-chain attribution supports the rumor."),
        ("An anonymous chat screenshot confirms a surprise $50B tech buyout before Friday close", "Unverified", "Without SEC Form 8-K filings or board confirmation, unauthenticated screenshots remain unverified."),
        ("Unpublished lab claims assert a solid-state car battery delivers 5,000 km on a 2-minute charge", "Unverified", "No peer-reviewed paper, third-party certification, or independent test data confirms the claim."),
        ("Social media rumors state Port of Rotterdam will close all container berths for 60 days", "Unverified", "Port of Rotterdam Authority operational bulletins report normal transit and zero corroboration."),
        ("Unconfirmed posts allege a universal zero-day breaks every TLS 1.3 handshake globally", "Unverified", "No CVE advisory, CISA alert, or reproducible proof-of-concept backs a universal TLS 1.3 break."),
        ("A deleted forum thread claimed divers recovered an intact 16th-century galleon last night", "Unverified", "No maritime museum, official expedition log, or archaeological authority corroborates the post."),
        ("Unattributed leaks claim a G7 central bank will abolish physical cash by next quarter", "Unverified", "No central bank minutes, legislative bill, or official press release confirms any cash phase-out."),
        ("An anonymous blog asserts a major stablecoin issuer lost 70% of Treasury reserves overnight", "Unverified", "No custodian notice, regulator halt, or audited reserve report substantiates the anonymous claim."),
        ("Unconfirmed shipping chatter says Suez Canal transit fees will triple starting Monday", "Unverified", "The Suez Canal Authority has issued no tariff circular announcing a threefold fee increase."),
        ("A private Telegram message claims an asteroid will disrupt GPS satellites next weekend", "Unverified", "NASA Planetary Defense and NOAA space weather bulletins show no such asteroid hazard."),
        ("Rumors circulate that a major cloud provider suffered permanent erasure of European S3 buckets", "Unverified", "Official status dashboards and incident reports show no confirmation of permanent data loss."),
        ("An unverified memo claims global semiconductor foundry capacity will halt for 90 days", "Unverified", "Foundry investor disclosures and industry trackers contain no confirmation of a 90-day shutdown."),
        ("Anonymous traders claim a Tier-1 exchange froze all institutional withdrawals this morning", "Unverified", "Exchange status pages and on-chain hot-wallet outflows show active processing with no freeze confirmation."),
        ("Unattributed posts say a hidden clause in next week's trade pact bans open-source cryptography", "Unverified", "Published draft treaty texts and legislative records contain no corroborated evidence of such a clause."),
        ("A viral audio clip allegedly captures an executive admitting hidden billion-dollar derivatives losses", "Unverified", "Forensic authentication, corporate filings, and independent reporting have not verified the clip."),
        ("Unconfirmed reports state an undersea cable cut severed 90% of transatlantic internet traffic", "Unverified", "Submarine cable operators and BGP routing monitors report no corroborated 90% capacity outage."),
        ("Anonymous forum posts claim a novel room-temperature superconductor entered mass factory production", "Unverified", "No independent laboratory replication or industrial certification confirms mass production."),
        ("Unverified rumors allege a sovereign debt moratorium was signed secretly at midnight", "Unverified", "Finance ministry announcements, IMF updates, and bond trustees report no signed moratorium."),
        ("A deleted social post claimed a major airline grounded its entire long-haul fleet indefinitely", "Unverified", "Aviation tracking feeds and official airline notices show normal scheduled departures."),
        ("Unattributed chatter claims an undisclosed hardware backdoor disables all ARM servers on Dec 31", "Unverified", "No silicon vendor advisory, CERT bulletin, or reproducible hardware erratum verifies the claim."),
    ]

    fact_items = []
    for i, (country, cap, wrong_city) in enumerate(capitals):
        if i % 2 == 0:
            claim = f"The official capital city of {country} is {cap}."
            v_g, v_b = "True", "False"
            r_g = f"Geographic records confirm {cap} is the capital of {country}, whereas {wrong_city} is a separate major city."
            r_b = f"Geographic records state {wrong_city} is the capital of {country}, whereas {cap} is not the capital."
        else:
            claim = f"The official capital city of {country} is {wrong_city}."
            v_g, v_b = "False", "True"
            r_g = f"Geographic records confirm {cap} is the capital of {country}, whereas {wrong_city} is not the capital."
            r_b = f"Geographic records state {wrong_city} is the capital of {country}, whereas {cap} is a separate major city."
        fact_items.append((claim, v_g, v_b, r_g, r_b, "https://www.cia.gov/the-world-factbook/"))

    for i, (el, sym, anum, wsym) in enumerate(elements):
        if i % 2 == 0:
            c1 = f"The IUPAC chemical symbol for {el} is {sym}."
            vg1, vb1 = "True", "False"
            rg1 = f"The periodic table assigns symbol {sym} to {el} (atomic number {anum}), not {wsym}."
            rb1 = f"The periodic table assigns symbol {wsym} to {el} (atomic number {anum}), not {sym}."
        else:
            c1 = f"The IUPAC chemical symbol for {el} is {wsym}."
            vg1, vb1 = "False", "True"
            rg1 = f"The periodic table assigns symbol {sym} to {el} (atomic number {anum}), not {wsym}."
            rb1 = f"The periodic table assigns symbol {wsym} to {el} (atomic number {anum}), not {sym}."
        fact_items.append((c1, vg1, vb1, rg1, rb1, "https://iupac.org/what-we-do/periodic-table-of-elements/"))

        wanum = anum + (7 if anum < 70 else -9)
        if i % 2 == 1:
            c2 = f"The atomic number (proton count) of {el} ({sym}) is {anum}."
            vg2, vb2 = "True", "False"
            rg2 = f"IUPAC atomic data lists {el} ({sym}) at proton count {anum}, rather than {wanum}."
            rb2 = f"IUPAC atomic data lists {el} ({sym}) at proton count {wanum}, rather than {anum}."
        else:
            c2 = f"The atomic number (proton count) of {el} ({sym}) is {wanum}."
            vg2, vb2 = "False", "True"
            rg2 = f"IUPAC atomic data lists {el} ({sym}) at proton count {anum}, rather than {wanum}."
            rb2 = f"IUPAC atomic data lists {el} ({sym}) at proton count {wanum}, rather than {anum}."
        fact_items.append((c2, vg2, vb2, rg2, rb2, "https://iupac.org/what-we-do/periodic-table-of-elements/"))

    for i, (subj, right_def, wrong_def) in enumerate(standards):
        if i % 2 == 0:
            c1 = f"In standard technical specifications, {subj} corresponds to {right_def}."
            vg1, vb1 = "True", "False"
            rg1 = f"Specification records define {subj} as {right_def}, distinct from {wrong_def}."
            rb1 = f"Specification records define {subj} as {wrong_def}, distinct from {right_def}."
        else:
            c1 = f"In standard technical specifications, {subj} corresponds to {wrong_def}."
            vg1, vb1 = "False", "True"
            rg1 = f"Specification records define {subj} as {right_def}, distinct from {wrong_def}."
            rb1 = f"Specification records define {subj} as {wrong_def}, distinct from {right_def}."
        fact_items.append((c1, vg1, vb1, rg1, rb1, "https://www.rfc-editor.org/"))

        if i % 2 == 1:
            c2 = f"Primary engineering documentation confirms {subj} denotes {right_def}."
            vg2, vb2 = "True", "False"
            rg2 = f"Documentation verifies {subj} maps to {right_def} rather than {wrong_def}."
            rb2 = f"Documentation verifies {subj} maps to {wrong_def} rather than {right_def}."
        else:
            c2 = f"Primary engineering documentation confirms {subj} denotes {wrong_def}."
            vg2, vb2 = "False", "True"
            rg2 = f"Documentation verifies {subj} maps to {right_def} rather than {wrong_def}."
            rb2 = f"Documentation verifies {subj} maps to {wrong_def} rather than {right_def}."
        fact_items.append((c2, vg2, vb2, rg2, rb2, "https://www.iana.org/"))

    for i, (subj, right_val, wrong_val) in enumerate(milestones):
        if i % 2 == 0:
            c1 = f"Historical and official registry records identify {subj} as {right_val}."
            vg1, vb1 = "True", "False"
            rg1 = f"Verified archives record {subj} as {right_val}, not {wrong_val}."
            rb1 = f"Verified archives record {subj} as {wrong_val}, not {right_val}."
        else:
            c1 = f"Historical and official registry records identify {subj} as {wrong_val}."
            vg1, vb1 = "False", "True"
            rg1 = f"Verified archives record {subj} as {right_val}, not {wrong_val}."
            rb1 = f"Verified archives record {subj} as {wrong_val}, not {right_val}."
        fact_items.append((c1, vg1, vb1, rg1, rb1, "https://www.iso.org/"))

        if i % 2 == 1:
            c2 = f"Public reference registries establish {right_val} for {subj}."
            vg2, vb2 = "True", "False"
            rg2 = f"Reference sources establish {right_val} for {subj} instead of {wrong_val}."
            rb2 = f"Reference sources establish {wrong_val} for {subj} instead of {right_val}."
        else:
            c2 = f"Public reference registries establish {wrong_val} for {subj}."
            vg2, vb2 = "False", "True"
            rg2 = f"Reference sources establish {right_val} for {subj} instead of {wrong_val}."
            rb2 = f"Reference sources establish {wrong_val} for {subj} instead of {right_val}."
        fact_items.append((c2, vg2, vb2, rg2, rb2, "https://www.britannica.com/"))

    for i, (subj, right_val, wrong_val) in enumerate(world_science):
        if i % 2 == 0:
            c1 = f"Standard scientific references identify the {subj} as {right_val}."
            vg1, vb1 = "True", "False"
            rg1 = f"Primary reference handbooks specify {right_val} for the {subj}, not {wrong_val}."
            rb1 = f"Primary reference handbooks specify {wrong_val} for the {subj}, not {right_val}."
        else:
            c1 = f"Standard scientific references identify the {subj} as {wrong_val}."
            vg1, vb1 = "False", "True"
            rg1 = f"Primary reference handbooks specify {right_val} for the {subj}, not {wrong_val}."
            rb1 = f"Primary reference handbooks specify {wrong_val} for the {subj}, not {right_val}."
        fact_items.append((c1, vg1, vb1, rg1, rb1, "https://www.nist.gov/"))

        if i % 2 == 1:
            c2 = f"Authoritative institutional records list {right_val} as the {subj}."
            vg2, vb2 = "True", "False"
            rg2 = f"Authoritative sources confirm {right_val} rather than {wrong_val} for the {subj}."
            rb2 = f"Authoritative sources confirm {wrong_val} rather than {right_val} for the {subj}."
        else:
            c2 = f"Authoritative institutional records list {wrong_val} as the {subj}."
            vg2, vb2 = "False", "True"
            rg2 = f"Authoritative sources confirm {right_val} rather than {wrong_val} for the {subj}."
            rb2 = f"Authoritative sources confirm {wrong_val} rather than {right_val} for the {subj}."
        fact_items.append((c2, vg2, vb2, rg2, rb2, "https://www.britannica.com/"))

    for i, (cl, vg, rg) in enumerate(nuance_100):
        other_cl, other_v, other_rg = nuance_100[(i + 23) % len(nuance_100)]
        fact_items.append((cl, vg, other_v, rg, other_rg, "https://www.oecd.org/", other_cl, other_v))

    prompt_templates = [
        lambda cl, d: f"Fact-check this claim ({d}): \"{cl}\"",
        lambda cl, d: f'Canonical ask FACT_CHECK_GENERAL: {{"claim":"{cl}"}} — return verdict, reasoning, and sources ({d}).',
        lambda cl, d: f"Verify factual accuracy for editorial audit {d}: {cl}",
        lambda cl, d: f"Grade the statement below as True, Mostly True, Mixed, Misleading, Mostly False, False, or Unverified [{d}]: {cl}",
        lambda cl, d: f"Compliance desk {d}: assess whether the following statement is accurate and cite primary reasoning: {cl}",
        lambda cl, d: f"Evidence review {d}: evaluate the truth value of \"{cl}\" with concise justification.",
        lambda cl, d: f"Intelligence brief check {d}: determine verdict and factual rationale for claim: {cl}",
        lambda cl, d: f"Public archive verification {d}: inspect claim \"{cl}\" and output structured verdict.",
        lambda cl, d: f"Newsroom triage {d}: verify claim accuracy and return canonical verdict for: {cl}",
        lambda cl, d: f"Research validation {d}: check whether the statement holds up against primary records — {cl}",
        lambda cl, d: f"Due diligence screen {d}: classify claim truthfulness and explain key evidence: {cl}",
    ]

    def render_ans(claim: str, verdict: str, reasoning: str, src: str, fmt: int) -> str:
        if fmt == 0:
            return json.dumps({"claim": claim, "verdict": verdict, "reasoning": reasoning, "sources": [src]})
        elif fmt == 1:
            return f"Verdict: {verdict}. Claim: {claim} Reasoning: {reasoning} Source: {src}"
        elif fmt == 2:
            return f'{{"verdict": "{verdict}", "claim": "{claim}"}} — {reasoning} (Source: {src})'
        else:
            return f"Claim: {claim} | Verdict: {verdict} | Analysis: {reasoning} | Evidence: {src}"

    rows = []
    for i in range(min(n_pairs, len(fact_items))):
        raw = fact_items[i]
        desk_id = f"FCK-{30000 + i * 7}"
        claim, v_good, v_bad, r_good, r_bad, src = raw[:6]
        prompt = prompt_templates[(i * 3) % len(prompt_templates)](claim, desk_id)
        if len(raw) == 8:
            other_claim, other_v = raw[6], raw[7]
            failure_tag = "nuanced_wrong_claim_or_verdict"
            r_cl, r_v, r_rs, r_src = other_claim, other_v, r_bad, src
        else:
            mode = i % 3
            if mode == 0:
                failure_tag = "verdict_and_fact_inversion"
                r_cl, r_v, r_rs, r_src = claim, v_bad, r_bad, src
            elif mode == 1:
                failure_tag = "verdict_reasoning_contradiction"
                r_cl, r_v, r_rs, r_src = claim, v_bad, r_good, src
            else:
                failure_tag = "wrong_claim_substitution"
                donor = fact_items[(i + 17) % 500]
                r_cl, r_v, r_rs, r_src = donor[0], donor[1], donor[3], donor[5]
        fmt = i % 4
        chosen = render_ans(claim, v_good, r_good, src, fmt)
        rejected = render_ans(r_cl, r_v, r_rs, r_src, fmt)
        chosen, rejected = _balance_pair_length(chosen, rejected, i, f" Audit {desk_id}.", f" Audit {desk_id}.")
        rows.append({
            "intent_id": "FACT_CHECK_GENERAL",
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected,
            "label_kind": "preference_pair",
            "failure_mode": failure_tag,
        })
    return rows


CORPUS_BUILDERS = {
    "LIQUIDITY_DEPTH_VERIFY": build_liquidity_corpus,
    "MACRO_ECONOMIC_INDICATOR": build_macro_corpus,
    "FACT_CHECK": build_fact_check_corpus,
}

DATASETS = {}
hp_gate = Hyperparams(seed=RANDOM_SEED, eval_split=0.15)

for fam in TARGET_INTENTS:
    rows = CORPUS_BUILDERS[fam](n_pairs=PAIRS_PER_INTENT, seed=RANDOM_SEED)
    gate_report = gate_dataset(rows, hp_gate)
    print(f"\n================ {fam} ({len(rows)} pairs) ================")
    print(format_gate(gate_report))
    if not gate_report["ok"]:
        raise RuntimeError(f"Dataset gate FAILED for {fam}: {gate_report['failures']}")

    train_rows, val_rows = split_pairs(rows, eval_frac=0.15, seed=RANDOM_SEED)
    fam_dir = DATA_DIR / fam
    fam_dir.mkdir(parents=True, exist_ok=True)
    for split_name, split_rows in (("full", rows), ("train", train_rows), ("val", val_rows)):
        (fam_dir / f"{split_name}.jsonl").write_text(
            "\n".join(json.dumps(r, ensure_ascii=False) for r in split_rows) + "\n",
            encoding="utf-8",
        )
    modes = Counter(r["failure_mode"] for r in rows)
    print(f"Split: train={len(train_rows)} (85%) | val={len(val_rows)} (15%)")
    print(f"Failure mode balance: {dict(modes)}")
    DATASETS[fam] = {
        "train": train_rows,
        "val": val_rows,
        "full_path": fam_dir / "full.jsonl",
        "train_path": fam_dir / "train.jsonl",
        "val_path": fam_dir / "val.jsonl",
        "gate_report": gate_report,
    }


## Cell 4 — GPU ModernBERT-Base LoRA Training + Hyperparameter Sweep + Early Stopping
Trains plain PEFT LoRA (`target_modules=["Wqkv", "Wo", "Wi"]`, `bias="none"`, `modules_to_save=["score", "classifier"]`) on the pinned `ModernBERT-base` backbone using Telegraph's exact `RenderForJudgment` prompt format (`"### Task:\n{prompt}\n\n### Answer:\n{answer}"`) and Bradley-Terry pairwise preference loss (`-logsigmoid(r_chosen - r_rejected)`).

- Sweeps **Candidate A (`rank=8, alpha=32`)** vs **Candidate B (`rank=16, alpha=32`)** when `RUN_HPARAM_SWEEP = True`.
- Uses early stopping on held-out `val_loss` and restores the exact best checkpoint (`adapter/` + `reward_head.pt`).
- Reports **held-out pairwise accuracy**, **mean/median reward margin**, **Spearman correlation**, **margin over length baseline**, and **per-failure-mode accuracy breakdown**.


In [ ]:
# @title Train ModernBERT-Base LoRA Reward Model (Sweep + Early Stopping + Detailed Eval)
import gc
import json
import math
import random
import shutil
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from float_runtime.train_adapter import (
    Hyperparams,
    PairCollator,
    build_model,
    bt_loss,
    render,
    save_reward_head,
)


def _spearman_corr(x: list[float], y: list[float]) -> float:
    if len(x) < 2:
        return 0.0
    rx = np.argsort(np.argsort(np.asarray(x, dtype=np.float64))).astype(np.float64)
    ry = np.argsort(np.argsort(np.asarray(y, dtype=np.float64))).astype(np.float64)
    rx -= rx.mean()
    ry -= ry.mean()
    denom = float(np.sqrt((rx * rx).sum() * (ry * ry).sum()))
    return round(float((rx * ry).sum() / denom), 4) if denom > 1e-12 else 0.0


def detailed_evaluate(model, tok, pairs: list[dict], max_length: int, eval_batch: int) -> dict:
    device = next(model.parameters()).device
    model.eval()
    loader = DataLoader(pairs, batch_size=eval_batch, shuffle=False, collate_fn=PairCollator(tok, max_length))
    chosen_scores, rejected_scores, diffs = [], [], []
    loss_sum = 0.0
    with torch.no_grad():
        for b in loader:
            rc = model(input_ids=b["chosen_input_ids"].to(device), attention_mask=b["chosen_attention_mask"].to(device)).logits.squeeze(-1).float()
            rr = model(input_ids=b["rejected_input_ids"].to(device), attention_mask=b["rejected_attention_mask"].to(device)).logits.squeeze(-1).float()
            loss_sum += bt_loss(rc, rr, b["margin"].to(device)).item() * rc.size(0)
            chosen_scores.extend(rc.cpu().tolist())
            rejected_scores.extend(rr.cpu().tolist())
            diffs.extend((rc - rr).cpu().tolist())

    n = len(diffs)
    acc = sum(1 for d in diffs if d > 0.0) / max(1, n)
    mean_margin = float(np.mean(diffs)) if diffs else 0.0
    median_margin = float(np.median(diffs)) if diffs else 0.0
    spearman = _spearman_corr(chosen_scores + rejected_scores, [1.0] * n + [0.0] * n)

    # Length baseline on val split
    longer = sum(1 for p in pairs if len(p["chosen"]) > len(p["rejected"]))
    ties = sum(1 for p in pairs if len(p["chosen"]) == len(p["rejected"]))
    len_acc = (longer + 0.5 * ties) / max(1, n)
    length_baseline = max(len_acc, 1.0 - len_acc)

    # Per-failure-mode accuracy breakdown
    by_mode = defaultdict(list)
    for p, d in zip(pairs, diffs):
        by_mode[p.get("failure_mode", "general")].append(1.0 if d > 0.0 else 0.0)
    per_mode_acc = {k: round(float(np.mean(v)), 4) for k, v in sorted(by_mode.items())}

    return {
        "n_eval": n,
        "pairwise_accuracy": round(acc, 4),
        "val_loss": round(loss_sum / max(1, n), 6),
        "mean_margin": round(mean_margin, 4),
        "median_margin": round(median_margin, 4),
        "spearman": spearman,
        "length_baseline": round(length_baseline, 4),
        "beats_length_baseline": acc >= length_baseline + 0.10,
        "verdict": "PASS" if (acc >= 0.65 and acc >= length_baseline + 0.10) else "FAIL",
        "per_failure_mode_accuracy": per_mode_acc,
    }


def train_one_candidate(fam: str, cand_name: str, cfg: dict, train_pairs: list[dict], val_pairs: list[dict]) -> dict:
    t0 = time.time()
    hp = Hyperparams(
        max_length=GPU_PROFILE["max_length"],
        batch_size=GPU_PROFILE["batch_size"],
        grad_accum=GPU_PROFILE["grad_accum"],
        grad_checkpointing=GPU_PROFILE["grad_checkpointing"],
        lora_rank=cfg["rank"],
        lora_alpha=cfg["alpha"],
        lora_dropout=cfg["dropout"],
        epochs=cfg["epochs"],
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        eval_batch=GPU_PROFILE["eval_batch"],
        seed=RANDOM_SEED,
        early_stopping=True,
        es_metric="val_loss",
        es_patience=2,
        es_min_delta=0.001,
    )

    random.seed(hp.seed)
    np.random.seed(hp.seed)
    torch.manual_seed(hp.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(hp.seed)

    run_dir = RUNS_DIR / fam / cand_name
    adapter_dir = run_dir / "adapter"
    if run_dir.exists():
        shutil.rmtree(run_dir)
    adapter_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n--- [{fam}] Candidate {cand_name}: rank={hp.lora_rank}, alpha={hp.lora_alpha}, lr={hp.lr}, epochs={hp.epochs} ---")
    model, tok = build_model(str(PINNED_MODEL_DIR), hp)
    device = torch.device(GPU_PROFILE["device"])
    amp_dtype = torch.bfloat16 if GPU_PROFILE["precision"] == "bf16" else (torch.float16 if GPU_PROFILE["precision"] == "fp16" else torch.float32)
    model.to(device)

    loader = DataLoader(train_pairs, batch_size=hp.batch_size, shuffle=True, collate_fn=PairCollator(tok, hp.max_length))
    params = [p for p in model.parameters() if p.requires_grad]
    optim = torch.optim.AdamW(params, lr=hp.lr, weight_decay=hp.weight_decay)
    steps_per_epoch = max(1, math.ceil(len(loader) / hp.grad_accum))
    total_steps = steps_per_epoch * hp.epochs
    warmup_steps = max(1, int(total_steps * 0.08))
    sched = torch.optim.lr_scheduler.LambdaLR(
        optim,
        lambda s: min(1.0, (s + 1) / warmup_steps)
        if s < warmup_steps
        else max(0.05, 0.5 * (1.0 + math.cos(math.pi * (s - warmup_steps) / max(1, total_steps - warmup_steps)))),
    )
    use_scaler = (device.type == "cuda" and amp_dtype == torch.float16)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    best_loss = math.inf
    best_acc = -1.0
    best_epoch = -1
    best_eval = None
    no_improve = 0
    history = []

    for epoch in range(hp.epochs):
        model.train()
        running = 0.0
        optim.zero_grad(set_to_none=True)
        for i, b in enumerate(loader):
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=(device.type == "cuda")):
                rc = model(input_ids=b["chosen_input_ids"].to(device), attention_mask=b["chosen_attention_mask"].to(device)).logits.squeeze(-1).float()
                rr = model(input_ids=b["rejected_input_ids"].to(device), attention_mask=b["rejected_attention_mask"].to(device)).logits.squeeze(-1).float()
                loss = bt_loss(rc, rr, b["margin"].to(device)) / hp.grad_accum
            if use_scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            running += loss.item() * hp.grad_accum

            if ((i + 1) % hp.grad_accum == 0) or ((i + 1) == len(loader)):
                if use_scaler:
                    scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                if use_scaler:
                    scaler.step(optim)
                    scaler.update()
                else:
                    optim.step()
                sched.step()
                optim.zero_grad(set_to_none=True)

        train_loss = running / max(1, len(loader))
        ev = detailed_evaluate(model, tok, val_pairs, hp.max_length, hp.eval_batch)
        history.append({"epoch": epoch, "train_loss": round(train_loss, 6), **ev})
        print(
            f"  epoch {epoch + 1}/{hp.epochs} | train_loss={train_loss:.4f} | "
            f"val_loss={ev['val_loss']:.4f} | val_acc={ev['pairwise_accuracy']:.4f} | "
            f"margin={ev['mean_margin']:+.3f} | spearman={ev['spearman']:.4f}"
        )

        is_better = (
            (ev["pairwise_accuracy"] > best_acc + 1e-4)
            or (abs(ev["pairwise_accuracy"] - best_acc) <= 1e-4 and ev["val_loss"] < best_loss - hp.es_min_delta)
        )
        if is_better or best_epoch < 0:
            best_acc = ev["pairwise_accuracy"]
            best_loss = ev["val_loss"]
            best_epoch = epoch
            best_eval = ev
            no_improve = 0
            model.save_pretrained(adapter_dir)
            tok.save_pretrained(adapter_dir)
            save_reward_head(model, adapter_dir)
        else:
            no_improve += 1

        if train_loss < 1e-4:
            print(f"  Early stop (loss floor reached); keeping best epoch {best_epoch + 1}.")
            break
        if no_improve > hp.es_patience:
            print(f"  Early stop ({no_improve} epochs without gain); keeping best epoch {best_epoch + 1}.")
            break

    elapsed = round(time.time() - t0, 1)
    result = {
        "intent": fam,
        "candidate": cand_name,
        "base_model": BASE_MODEL_ID,
        "tokenizer_sha256": PINNED_HASHES["tokenizer.json"],
        "adapter_dir": str(adapter_dir),
        "reward_head": str(adapter_dir / "reward_head.pt"),
        "train_pairs": len(train_pairs),
        "val_pairs": len(val_pairs),
        "best_epoch": best_epoch + 1,
        "elapsed_sec": elapsed,
        "rank": hp.lora_rank,
        "alpha": hp.lora_alpha,
        "lr": hp.lr,
        "dropout": hp.lora_dropout,
        "max_length": hp.max_length,
        **best_eval,
        "history": history,
    }
    (run_dir / "result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")

    # Free GPU memory cleanly between candidates
    del model, optim, sched, loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


SWEEP_CONFIGS = [
    ("r8_a32", {"rank": 8, "alpha": 32, "lr": 1.2e-4, "dropout": 0.10, "weight_decay": 0.01, "epochs": 5}),
]
if RUN_HPARAM_SWEEP:
    SWEEP_CONFIGS.append(
        ("r16_a32", {"rank": 16, "alpha": 32, "lr": 8.0e-5, "dropout": 0.15, "weight_decay": 0.02, "epochs": 5})
    )

CHAMPION_RUNS = {}
for fam in TARGET_INTENTS:
    ds = DATASETS[fam]
    candidates = []
    for cand_name, cfg in SWEEP_CONFIGS:
        res = train_one_candidate(fam, cand_name, cfg, ds["train"], ds["val"])
        candidates.append(res)

    # Pick champion: highest held-out pairwise accuracy, tie-break lowest val_loss, then highest mean_margin
    best = sorted(
        candidates,
        key=lambda r: (-r["pairwise_accuracy"], r["val_loss"], -r["mean_margin"]),
    )[0]

    # Write official Telegraph TrainingRecord (pkg/wasm/runtime/adaptermeta.go)
    spec = INTENT_SPECS[fam]
    training_record = {
        "family": fam,
        "trainer": f"{AUTHOR_ORG}-colab-gpu-lora-v1",
        "backbone": BASE_MODEL_ID,
        "backbone_version": FOUNDATION_TAG,
        "children": [
            {
                "intent": spec["child_intent"],
                "dataset_version": f"{AUTHOR_ORG.lower()}-{fam.lower()}-600v1",
                "pairs": len(ds["train"]) + len(ds["val"]),
                "accuracy": best["pairwise_accuracy"],
                "spearman": best["spearman"],
                "held_out": True,
            }
        ],
    }
    best_run_dir = Path(best["adapter_dir"]).parent
    tr_path = best_run_dir / "training_record.json"
    tr_path.write_text(json.dumps(training_record, indent=2), encoding="utf-8")

    CHAMPION_RUNS[fam] = {
        "best": best,
        "all_candidates": candidates,
        "best_run_dir": best_run_dir,
        "training_record_path": tr_path,
        "training_record": training_record,
    }

    print(f"\n================ CHAMPION SELECTED FOR {fam}: {best['candidate']} ================")
    print(f"  Held-out Pairwise Accuracy : {best['pairwise_accuracy']:.2%} (length baseline: {best['length_baseline']:.2%})")
    print(f"  Held-out BT Loss / Margin  : loss={best['val_loss']:.4f}, mean_margin={best['mean_margin']:+.3f}, median={best['median_margin']:+.3f}")
    print(f"  Spearman Correlation       : {best['spearman']:.4f}")
    print(f"  Per-Failure-Mode Accuracy  : {json.dumps(best['per_failure_mode_accuracy'])}")
    if best["verdict"] != "PASS":
        raise RuntimeError(f"Training evaluation gate FAILED for {fam}: {best}")


## Cell 5 — Export `.safetensors` (Bug `#223` Tokenizer Parity) + Full Stage-1 Validator + Optional WASM Double-Score Check
1. Restores the exact verified pinned `tokenizer.json` (`67bc76f8...`) over `<best_run>/adapter/tokenizer.json` (undoing `transformers` `save_pretrained` byte re-serialization — Bug `#223`!).
2. Calls the **unmodified official** `python -m float_runtime.export_adapter` to produce `<INTENT>_upswing_champion.safetensors`.
3. Runs a complete **Stage-1 Structural Validator** (100% parity with `pkg/wasm/runtime/lora_adapter.go` + `track3_evalforge/adapter_spec.mjs`: **178 F32 tensors**, exact shapes, contiguous offsets, zero trailing bytes, all finite float32 values, pinned `foundation_hash` & `tokenizer_hash`, valid `training` record, `<= 64 MiB`, Ethereum **Keccak-256 `wasmHash`** + **SHA-256**).
4. If `VERIFY_ON_WASM_FOUNDATION = True`, downloads `foundation_lora_fp32.onnx` + `foundation_engine.wasm` + Go 1.25.7 and executes `go run ./cmd/adapterbuild lora` to confirm **bit-exact WASM determinism** on the real validator engine!


In [ ]:
# @title Export Official lora-v1 .safetensors + Stage-1 Parity Audit + Optional WASM Bit-Exact Check
import hashlib
import json
import math
import os
import shutil
import struct
import subprocess
import sys
import urllib.request
from pathlib import Path

EXPECTED_TARGETS = {
    "attn.Wqkv": (768, 2304),
    "attn.Wo": (768, 768),
    "mlp.Wi": (768, 2304),
    "mlp.Wo": (1152, 768),
}
REQUIRED_META_KEYS = (
    "format_version", "abi", "class", "intent", "foundation_hash",
    "foundation_version", "tokenizer_hash", "rank", "alpha",
    "target_modules", "max_tokens",
)
KNOWN_META_KEYS = set(REQUIRED_META_KEYS) | {"training"}
ALLOWED_TARGET_MODULES = {"Wi", "Wo", "Wqkv"}


def validate_lora_package_stage1(pkg_path: Path, expected_intent: str, expected_class: str = "judgment") -> dict:
    """100% parity with Telegraph pkg/wasm/runtime/lora_adapter.go & track3_evalforge/adapter_spec.mjs."""
    raw = pkg_path.read_bytes()
    max_bytes = 64 * 1024 * 1024
    if len(raw) < 9:
        raise ValueError(f"bad_container: {len(raw)} bytes < 9")
    if len(raw) > max_bytes:
        raise ValueError(f"too_large: {len(raw)} > {max_bytes}")

    header_len = struct.unpack("<Q", raw[:8])[0]
    if header_len == 0 or header_len > (1 << 20):
        raise ValueError(f"bad_container: invalid header length {header_len}")
    if len(raw) < 8 + header_len:
        raise ValueError("truncated: header extends past EOF")

    header = json.loads(raw[8:8 + header_len].decode("utf-8"))
    meta = header.get("__metadata__")
    if not isinstance(meta, dict):
        raise ValueError("missing_metadata: __metadata__ missing")

    for k in REQUIRED_META_KEYS:
        if k not in meta or not isinstance(meta[k], str):
            raise ValueError(f"missing_metadata: missing string key {k!r}")
    for k in meta:
        if k not in KNOWN_META_KEYS and not k.startswith("x-"):
            raise ValueError(f"bad_header: unknown metadata key {k!r}")

    assert meta["format_version"] == "1", f"bad_version: {meta['format_version']}"
    assert meta["abi"] == "lora-v1", f"bad_abi: {meta['abi']}"
    assert meta["class"] == expected_class, f"class_mismatch: {meta['class']} != {expected_class}"
    assert meta["intent"] == expected_intent, f"intent_mismatch: {meta['intent']} != {expected_intent}"
    assert meta["foundation_hash"].lower() == PINNED_HASHES["foundation_lora_fp32.onnx"], "foundation_mismatch"
    assert meta["foundation_version"] == FOUNDATION_TAG, "foundation_version_mismatch"
    assert meta["tokenizer_hash"].lower() == PINNED_HASHES["tokenizer.json"], "tokenizer_mismatch (Bug #223!)"

    rank = int(meta["rank"])
    alpha = float(meta["alpha"])
    max_tokens = int(meta["max_tokens"])
    tmods = json.loads(meta["target_modules"])
    assert 1 <= rank <= 16, f"rank_too_large: {rank}"
    assert math.isfinite(alpha) and alpha > 0, f"bad_alpha: {alpha}"
    assert 1 <= max_tokens <= 512, f"bad_max_tokens: {max_tokens}"
    assert isinstance(tmods, list) and len(tmods) == len(set(tmods)) and set(tmods).issubset(ALLOWED_TARGET_MODULES)

    if "training" in meta:
        tr = json.loads(meta["training"])
        assert tr.get("family") == expected_intent and len(tr.get("children", [])) >= 1
        for ch in tr["children"]:
            assert ch["intent"] == expected_intent or ch["intent"].startswith(expected_intent + "_")
            assert ch["pairs"] >= 0 and 0.0 <= ch["accuracy"] <= 1.0 and -1.0 <= ch["spearman"] <= 1.0

    # Exact 178 F32 tensors tiled contiguously across 22 layers + reward head
    expected_shapes = {}
    for layer in range(22):
        for target, (in_dim, out_dim) in EXPECTED_TARGETS.items():
            expected_shapes[f"lora.{layer}.{target}.A"] = [in_dim, rank]
            expected_shapes[f"lora.{layer}.{target}.B"] = [rank, out_dim]
    expected_shapes["head.weight"] = [768, 1]
    expected_shapes["head.bias"] = [1]

    tensors = []
    for name, info in header.items():
        if name == "__metadata__":
            continue
        tensors.append((info["data_offsets"][0], info["data_offsets"][1], name, info["dtype"], info["shape"]))
    tensors.sort(key=lambda x: (x[0], x[2]))

    assert len(tensors) == 178 and len(tensors) == len(expected_shapes), f"bad_shape: got {len(tensors)} tensors, expected 178"
    data = raw[8 + header_len:]
    offset = 0
    seen = set()
    for begin, end, name, dtype, shape in tensors:
        assert name in expected_shapes and name not in seen, f"unexpected/duplicate tensor {name}"
        seen.add(name)
        assert dtype == "F32", f"bad_dtype: {name} is {dtype}"
        assert shape == expected_shapes[name], f"bad_shape: {name} {shape} != {expected_shapes[name]}"
        byte_len = 4 * math.prod(shape)
        assert begin == offset and (end - begin) == byte_len, f"bad_offsets: {name} [{begin},{end}] != [{offset},{offset + byte_len}]"
        assert end <= len(data), f"truncated: {name}"
        offset = end
    assert offset == len(data), f"trailing_bytes: {len(data) - offset}"

    # Verify 100% finite float32 values
    import numpy as np
    arr = np.frombuffer(data, dtype="<f4")
    assert bool(np.isfinite(arr).all()), "non_finite: NaN or Inf found in tensor data!"

    return {
        "valid": True,
        "path": str(pkg_path),
        "filename": pkg_path.name,
        "bytes": len(raw),
        "tensor_count": len(tensors),
        "rank": rank,
        "alpha": alpha,
        "max_tokens": max_tokens,
        "keccak256_wasm_hash": keccak256_hex(raw, prefix=True),
        "adapter_hash_no_prefix": keccak256_hex(raw, prefix=False),
        "sha256": hashlib.sha256(raw).hexdigest(),
        "metadata": meta,
    }


EXPORTED_PACKAGES = {}

for fam in TARGET_INTENTS:
    info = CHAMPION_RUNS[fam]
    best_run_dir = info["best_run_dir"]
    adapter_dir = best_run_dir / "adapter"
    tr_path = info["training_record_path"]

    # Automated Bug #223 Export Fix: overwrite save_pretrained tokenizer.json with exact pinned release bytes
    shutil.copy2(PINNED_TOKENIZER_PATH, adapter_dir / "tokenizer.json")
    assert sha256_file(adapter_dir / "tokenizer.json") == PINNED_HASHES["tokenizer.json"]

    # Ensure result.json reflects canonical base_model & pinned tokenizer_sha256 for official export_adapter.py
    res_path = best_run_dir / "result.json"
    res_data = json.loads(res_path.read_text(encoding="utf-8"))
    res_data["base_model"] = BASE_MODEL_ID
    res_data["tokenizer_sha256"] = PINNED_HASHES["tokenizer.json"]
    res_path.write_text(json.dumps(res_data, indent=2), encoding="utf-8")

    out_pkg = EXPORT_DIR / f"{fam}_upswing_champion.safetensors"
    cmd = [
        sys.executable, "-m", "float_runtime.export_adapter",
        "--run", str(best_run_dir),
        "--intent", fam,
        "--class", INTENT_SPECS[fam]["class"],
        "--foundation", str(FOUNDATION_JSON_PATH),
        "--training-record", str(tr_path),
        "--max-tokens", "512",
        "--out", str(out_pkg),
    ]
    print(f"\nRunning official export_adapter for {fam} ...")
    proc = subprocess.run(cmd, cwd=str(KIT_DIR), capture_output=True, text=True, check=False)
    print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"official export_adapter failed for {fam} (exit={proc.returncode})")

    stage1 = validate_lora_package_stage1(out_pkg, expected_intent=fam, expected_class=INTENT_SPECS[fam]["class"])
    EXPORTED_PACKAGES[fam] = stage1
    print(f"STAGE-1 PASS [{fam}]:")
    print(f"  File           : {stage1['filename']} ({stage1['bytes']:,} bytes, {stage1['tensor_count']} F32 tensors)")
    print(f"  Rank / Alpha   : r={stage1['rank']}, alpha={stage1['alpha']}, max_tokens={stage1['max_tokens']}")
    print(f"  Keccak-256     : {stage1['keccak256_wasm_hash']}   <-- registerWasm wasmHash!")
    print(f"  SHA-256        : {stage1['sha256']}")
    print(f"  Tokenizer Hash : {stage1['metadata']['tokenizer_hash']} (Bug #223 VERIFIED)")

# Optional: Full real-WASM double-score bit-exact verification via official cmd/adapterbuild lora
if VERIFY_ON_WASM_FOUNDATION:
    print("\n=== Running Official WASM Foundation Bit-Exact Verification (cmd/adapterbuild lora) ===")
    onnx_path = download_verified("foundation_lora_fp32.onnx", PINNED_HASHES["foundation_lora_fp32.onnx"])
    engine_path = download_verified("foundation_engine.wasm", PINNED_HASHES["foundation_engine.wasm"])

    go_bin = shutil.which("go")
    if not go_bin:
        go_tgz = WORK_ROOT / "go1.25.7.linux-amd64.tar.gz"
        if not go_tgz.exists():
            req = urllib.request.Request(
                "https://go.dev/dl/go1.25.7.linux-amd64.tar.gz",
                headers={"User-Agent": "Upswing-Labs-Colab-Forge"},
            )
            with urllib.request.urlopen(req, timeout=180) as r, go_tgz.open("wb") as f:
                shutil.copyfileobj(r, f)
        subprocess.run(["tar", "-C", "/usr/local", "-xzf", str(go_tgz)], check=True)
        os.environ["PATH"] = "/usr/local/go/bin:" + os.environ.get("PATH", "")
        go_bin = "/usr/local/go/bin/go"

    for fam, s1 in EXPORTED_PACKAGES.items():
        ab_cmd = [
            go_bin, "run", "./cmd/adapterbuild", "lora",
            "--intent", fam,
            "--package", s1["path"],
            "--foundation", str(FOUNDATION_JSON_PATH),
            "--foundation-onnx", str(onnx_path),
            "--engine", str(engine_path),
            "--engine-hash", PINNED_HASHES["foundation_engine.wasm"],
            "--out", str(EXPORT_DIR),
        ]
        ab = subprocess.run(ab_cmd, cwd=str(KIT_DIR), capture_output=True, text=True, check=False)
        print(ab.stderr)
        print(ab.stdout)
        if ab.returncode != 0:
            raise RuntimeError(f"adapterbuild lora WASM check failed for {fam}")
        s1["wasm_double_score_verified"] = True
        s1["adapterbuild_json"] = json.loads(ab.stdout.strip().splitlines()[-1])
        assert s1["adapterbuild_json"]["keccak256"].lower() == s1["keccak256_wasm_hash"].lower()
        print(f"WASM BIT-EXACT VERIFIED [{fam}]: score_hex={s1['adapterbuild_json'].get('score_hex')}")


## Cell 6 — Generate Bit-Exact `adapterexec` Deploy Kit (`Fly.io` / `Render` / `Cloudflare Tunnel`) + `training_evidence.json` + Auto-Download

> **CRITICAL PROTOCOL NOTE:** Telegraph validators reject `wasmRunnableUrl` endpoints that run PyTorch or native ONNX Runtime (`remote score mismatch`, e.g. Registration `#203` on `LIQUIDITY_DEPTH_VERIFY`). Your runnable executor **MUST** run Telegraph's official **`cmd/adapterexec`** (Go 1.25.7 + `foundation_lora_fp32.onnx` + `foundation_engine.wasm`) to match validator IEEE-754 float32 bits **bit-for-bit**!

This cell writes:
1. `training_evidence.json` — full reproducible metrics, hashes, sweep comparison, and Sepolia `registerWasm` parameters.
2. `adapterexec_deploy/` — drop-in `Dockerfile`, `fly.toml` (4 GB RAM), `render.yaml`, `run_local_cloudflare_tunnel.sh`, `verify_executor.py` (`check-executor.ts` parity verifier), and `register_instructions.md`.
3. `upswing_telegraph_lora_champions_bundle.zip` — everything archived neatly AND auto-downloaded to your PC via `google.colab.files.download`!


In [ ]:
# @title Package Production adapterexec Deploy Kit, Write training_evidence.json & Auto-Download
import datetime
import json
import os
import shutil
import subprocess
import urllib.request
import zipfile
from pathlib import Path

DEPLOY_DIR = EXPORT_DIR / "adapterexec_deploy"
if DEPLOY_DIR.exists():
    shutil.rmtree(DEPLOY_DIR)
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

primary_intent = TARGET_INTENTS[0]
primary_pkg_name = f"{primary_intent}_upswing_champion.safetensors"

# Copy trained .safetensors packages into adapterexec_deploy/models/
models_dir = DEPLOY_DIR / "models"
models_dir.mkdir(parents=True, exist_ok=True)
for fam, s1 in EXPORTED_PACKAGES.items():
    shutil.copy2(Path(s1["path"]), models_dir / Path(s1["path"]).name)

# 1. Production Multi-Stage Dockerfile (Go 1.25.7 official cmd/adapterexec + SHA-256 verified foundation)
dockerfile_txt = f"""# Upswing-Labs — Bit-Exact Telegraph V2 Evaluator Runnable Executor (cmd/adapterexec)
# Serves modernbert-base-lora16-v1 LoRA .safetensors through Telegraph's official WASM engine.
# Recommended RAM: >= 3 GB (foundation worker peaks at ~1.8 GB RAM).

FROM golang:1.25-bookworm AS builder
WORKDIR /src
RUN git clone --depth 1 https://github.com/telegraphprotocol/telegraph-wasm-baseline.git . && \\
    CGO_ENABLED=0 GOOS=linux go build -o /out/adapterexec ./cmd/adapterexec

FROM debian:bookworm-slim
RUN apt-get update && apt-get install -y --no-install-recommends ca-certificates curl && \\
    rm -rf /var/lib/apt/lists/*
WORKDIR /app
COPY --from=builder /out/adapterexec /usr/local/bin/adapterexec

# Download & SHA-256 verify exact canonical modernbert-base-lora16-v1 foundation assets
RUN mkdir -p /app/foundation && \\
    BASE="https://github.com/telegraphprotocol/telegraph-foundation/releases/download/{FOUNDATION_TAG}" && \\
    curl -fsSL --retry 3 -o /app/foundation/foundation_lora_fp32.onnx "$BASE/foundation_lora_fp32.onnx" && \\
    curl -fsSL --retry 3 -o /app/foundation/foundation_engine.wasm "$BASE/foundation_engine.wasm" && \\
    curl -fsSL --retry 3 -o /app/foundation/foundation.json "$BASE/foundation.json" && \\
    curl -fsSL --retry 3 -o /app/foundation/tokenizer.json "$BASE/tokenizer.json" && \\
    curl -fsSL --retry 3 -o /app/foundation/SHA256SUMS "$BASE/SHA256SUMS" && \\
    cd /app/foundation && sha256sum -c SHA256SUMS

COPY models/ /app/models/

ENV FOUNDATION_MODEL_PATH=/app/foundation/foundation_lora_fp32.onnx
ENV FOUNDATION_MODEL_HASH={PINNED_HASHES['foundation_lora_fp32.onnx']}
ENV FOUNDATION_MODEL_VERSION={FOUNDATION_TAG}
ENV FOUNDATION_ENGINE_WASM_PATH=/app/foundation/foundation_engine.wasm
ENV FOUNDATION_ENGINE_WASM_HASH={PINNED_HASHES['foundation_engine.wasm']}
ENV ADAPTER_PACKAGE=/app/models/{primary_pkg_name}
ENV PORT=8080

EXPOSE 8080
CMD ["sh", "-c", "exec /usr/local/bin/adapterexec --package \\"$ADAPTER_PACKAGE\\" --listen \\"0.0.0.0:${{PORT:-8080}}\\""]
"""
(DEPLOY_DIR / "Dockerfile").write_text(dockerfile_txt, encoding="utf-8")

# 2. Fly.io configuration (4 GB RAM, auto-TLS public HTTPS)
fly_toml = f"""app = "upswing-telegraph-adapterexec"
primary_region = "iad"

[build]
  dockerfile = "Dockerfile"

[env]
  PORT = "8080"
  ADAPTER_PACKAGE = "/app/models/{primary_pkg_name}"

[http_service]
  internal_port = 8080
  force_https = true
  auto_stop_machines = "off"
  auto_start_machines = true
  min_machines_running = 1

[[vm]]
  cpu_kind = "shared"
  cpus = 2
  memory = "4gb"
"""
(DEPLOY_DIR / "fly.toml").write_text(fly_toml, encoding="utf-8")

# 3. Render Docker Blueprint
render_yaml = f"""services:
  - type: web
    name: upswing-telegraph-adapterexec
    runtime: docker
    plan: standard
    dockerfilePath: ./Dockerfile
    envVars:
      - key: PORT
        value: "8080"
      - key: ADAPTER_PACKAGE
        value: "/app/models/{primary_pkg_name}"
"""
(DEPLOY_DIR / "render.yaml").write_text(render_yaml, encoding="utf-8")

# 4. Local Linux/macOS + Cloudflare Quick Tunnel launcher (SHA-256 verified downloads)
tunnel_sh = f"""#!/usr/bin/env bash
set -euo pipefail
# Runs official cmd/adapterexec locally on 127.0.0.1:8080 and exposes public HTTPS via cloudflared.
# Prerequisites: Go 1.25.7+ and cloudflared installed locally.

PKG="${{1:-models/{primary_pkg_name}}}"
ROOT="$(cd "$(dirname "$0")" && pwd)"
KIT_DIR="$ROOT/telegraph-wasm-baseline"
F_DIR="$ROOT/foundation"
mkdir -p "$F_DIR"

if [ ! -d "$KIT_DIR" ]; then
  git clone --depth 1 https://github.com/telegraphprotocol/telegraph-wasm-baseline.git "$KIT_DIR"
fi

BASE="https://github.com/telegraphprotocol/telegraph-foundation/releases/download/{FOUNDATION_TAG}"
for f in foundation_lora_fp32.onnx foundation_engine.wasm foundation.json tokenizer.json SHA256SUMS; do
  if [ ! -f "$F_DIR/$f" ]; then
    curl -fsSL --retry 3 -o "$F_DIR/$f" "$BASE/$f"
  fi
done
(cd "$F_DIR" && sha256sum -c SHA256SUMS)

export FOUNDATION_MODEL_PATH="$F_DIR/foundation_lora_fp32.onnx"
export FOUNDATION_MODEL_HASH="{PINNED_HASHES['foundation_lora_fp32.onnx']}"
export FOUNDATION_MODEL_VERSION="{FOUNDATION_TAG}"
export FOUNDATION_ENGINE_WASM_PATH="$F_DIR/foundation_engine.wasm"
export FOUNDATION_ENGINE_WASM_HASH="{PINNED_HASHES['foundation_engine.wasm']}"

echo "Starting official adapterexec on 127.0.0.1:8080 for $PKG ..."
(cd "$KIT_DIR" && go run ./cmd/adapterexec --package "$ROOT/$PKG" --listen 127.0.0.1:8080) &
EXEC_PID=$!
trap 'kill $EXEC_PID 2>/dev/null || true' EXIT
sleep 5

echo "Opening Cloudflare Quick Tunnel (copy the https://*.trycloudflare.com URL as wasmRunnableUrl) ..."
cloudflared tunnel --url http://127.0.0.1:8080
"""
(DEPLOY_DIR / "run_local_cloudflare_tunnel.sh").write_text(tunnel_sh, encoding="utf-8")

# 5. Exact check-executor.ts Python verification client (determinism + input_root + score identity echo)
verify_exec_py = f"""#!/usr/bin/env python3
\"\"\"Verifies any live wasmRunnableUrl against Telegraph V2 check-executor.ts contract.\"\"\"
import hashlib
import json
import math
import struct
import sys
import urllib.request

BATCH = [
    [50281, 4118, 17526, 27, 187, 18658, 436, 13814, 1818, 50282],
    [50281, 4118, 17526, 27, 187, 1276, 310, 253, 5347, 50282],
    [50281, 4118, 17526, 27, 187, 18658, 436, 13814, 1818, 50282],
]

def encode_float_input(ids):
    return '{{"input_ids":[' + ','.join(map(str, ids)) + '],"attention_mask":[' + ','.join(['1'] * len(ids)) + ']}}'

def batch_input_root(token_lists):
    leaves = sorted(f"{{i}}:{{hashlib.sha256(encode_float_input(ids).encode()).hexdigest()}}" for i, ids in enumerate(token_lists))
    return hashlib.sha256("\\n".join(leaves).encode()).hexdigest()

def u32_to_f32(u):
    return struct.unpack("<f", struct.pack("<I", u & 0xFFFFFFFF))[0]

def main():
    if len(sys.argv) < 3:
        print("Usage: python verify_executor.py <RUNNABLE_URL> <ADAPTER_KECCAK256_NO_0X>")
        raise SystemExit(2)
    url = sys.argv[1]
    adapter_hash = sys.argv[2].lower().removeprefix("0x")
    foundation_hash = "{PINNED_HASHES['foundation_lora_fp32.onnx']}"
    tokenizer_hash = "{PINNED_HASHES['tokenizer.json']}"
    root = batch_input_root(BATCH)

    payload = json.dumps({{
        "batch": [{{"tokens": t}} for t in BATCH],
        "foundation_hash": foundation_hash,
        "adapter_hash": adapter_hash,
        "tokenizer_hash": tokenizer_hash,
        "input_root": root,
    }}).encode("utf-8")

    req = urllib.request.Request(url, data=payload, method="POST", headers={{"Content-Type": "application/json"}})
    with urllib.request.urlopen(req, timeout=120) as resp:
        assert resp.status == 200, f"Expected HTTP 200, got {{resp.status}}"
        body = json.loads(resp.read().decode("utf-8"))

    res = body.get("results", [])
    assert len(res) == 3, f"Expected 3 results, got {{len(res)}}"
    assert res[0]["score"] == res[2]["score"], "Non-deterministic scores on identical inputs!"
    for idx, item in enumerate(res):
        u = item["score"]
        assert isinstance(u, int) and 0 <= u <= 0xFFFFFFFF
        f = u32_to_f32(u)
        assert math.isfinite(f), f"Non-finite float at index {{idx}}"
    assert body.get("foundation_hash") == foundation_hash, f"foundation_hash mismatch: {{body.get('foundation_hash')}}"
    assert body.get("adapter_hash") == adapter_hash, f"adapter_hash mismatch: {{body.get('adapter_hash')}}"
    assert body.get("tokenizer_hash") == tokenizer_hash, f"tokenizer_hash mismatch: {{body.get('tokenizer_hash')}}"
    assert body.get("input_root") == root, f"input_root mismatch: {{body.get('input_root')}} != {{root}}"

    print("EXECUTOR CONTRACT VERIFIED 100% PASS!")
    print(f"  scores (u32 -> f32): {{[(r['score'], round(u32_to_f32(r['score']), 6)) for r in res]}}")
    print(f"  input_root         : {{root}}")

if __name__ == "__main__":
    main()
"""
(DEPLOY_DIR / "verify_executor.py").write_text(verify_exec_py, encoding="utf-8")

# 6. Registration instructions (Sepolia Diamond 0x72a70e949362670Bf8bceAd24eFC5b8f950aa08b)
reg_lines = [
    "# Telegraph Protocol V2 — Sepolia `registerWasm` Champion Registration Checklist",
    "",
    "- **Chain**: Ethereum Sepolia (`chainId: 11155111`)",
    "- **Diamond Contract**: `0x72a70e949362670Bf8bceAd24eFC5b8f950aa08b`",
    "- **Function**: `registerWasm(bytes32 wasmHash, string wasmUrl, string intent, address feeAddress, string runnableUrl)`",
    "- **Dashboard**: `https://anjoman.ai/dashboard/eval`",
    "",
    "## Trained Packages Ready to Register",
    "",
]
for fam, s1 in EXPORTED_PACKAGES.items():
    best = CHAMPION_RUNS[fam]["best"]
    reg_lines += [
        f"### `{fam}` (`class: judgment`)",
        f"- **File**: `{s1['filename']}` ({s1['bytes']:,} bytes, rank `{s1['rank']}`, alpha `{s1['alpha']}`)",
        f"- **Held-out Accuracy**: `{best['pairwise_accuracy']:.2%}` | **Margin**: `{best['mean_margin']:+.3f}` | **Spearman**: `{best['spearman']:.4f}`",
        f"- **`wasmHash` (Keccak-256)**: `{s1['keccak256_wasm_hash']}`",
        f"- **SHA-256**: `{s1['sha256']}`",
        f"- **Host `wasmUrl`**: Commit `{s1['filename']}` to a public GitHub repo (or GitHub Release / IPFS) and copy the immutable raw HTTPS URL.",
        f"- **Deploy `runnableUrl`**: Deploy `adapterexec_deploy/` on Fly.io (`fly deploy`), Render, or local `run_local_cloudflare_tunnel.sh`, then verify with:",
        f"  ```bash",
        f"  python verify_executor.py https://<YOUR_RUNNABLE_URL> {s1['adapter_hash_no_prefix']}",
        f"  ```",
        "",
    ]
(DEPLOY_DIR / "register_instructions.md").write_text("\n".join(reg_lines), encoding="utf-8")

# 7. Write comprehensive training_evidence.json
evidence = {
    "schema": "upswing-telegraph-track3-lora-evidence-v1",
    "generated_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "organization": AUTHOR_ORG,
    "network": {
        "chain": "Ethereum Sepolia",
        "chain_id": 11155111,
        "diamond": "0x72a70e949362670Bf8bceAd24eFC5b8f950aa08b",
        "devnode": "https://devnode.telegraphprotocol.com",
    },
    "foundation": {
        "version": FOUNDATION_TAG,
        "base_model": BASE_MODEL_ID,
        "base_model_revision": BASE_MODEL_REV,
        "foundation_hash": PINNED_HASHES["foundation_lora_fp32.onnx"],
        "engine_wasm_hash": PINNED_HASHES["foundation_engine.wasm"],
        "tokenizer_hash": PINNED_HASHES["tokenizer.json"],
        "bug_223_mitigated": True,
    },
    "hardware": GPU_PROFILE,
    "targets": {},
}
for fam in TARGET_INTENTS:
    evidence["targets"][fam] = {
        "spec": INTENT_SPECS[fam],
        "dataset_gate": DATASETS[fam]["gate_report"],
        "champion_metrics": {k: v for k, v in CHAMPION_RUNS[fam]["best"].items() if k != "history"},
        "sweep_summary": [
            {
                "candidate": c["candidate"],
                "rank": c["rank"],
                "alpha": c["alpha"],
                "pairwise_accuracy": c["pairwise_accuracy"],
                "val_loss": c["val_loss"],
                "mean_margin": c["mean_margin"],
                "spearman": c["spearman"],
                "elapsed_sec": c["elapsed_sec"],
            }
            for c in CHAMPION_RUNS[fam]["all_candidates"]
        ],
        "stage1_package": EXPORTED_PACKAGES[fam],
    }

evidence_path = EXPORT_DIR / "training_evidence.json"
evidence_path.write_text(json.dumps(evidence, indent=2), encoding="utf-8")
print(f"Wrote {evidence_path}")

# 8. Archive full bundle zip
bundle_zip = EXPORT_DIR / "upswing_telegraph_lora_champions_bundle.zip"
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(evidence_path, "training_evidence.json")
    for fam, s1 in EXPORTED_PACKAGES.items():
        zf.write(Path(s1["path"]), Path(s1["path"]).name)
    for fp in sorted(DEPLOY_DIR.rglob("*")):
        if fp.is_file():
            zf.write(fp, str(Path("adapterexec_deploy") / fp.relative_to(DEPLOY_DIR)))
print(f"Created {bundle_zip} ({bundle_zip.stat().st_size:,} bytes)")

# 9. Print crisp summary table
print("\n================ FINAL UPSWING CHAMPION SUMMARY ================")
for fam, s1 in EXPORTED_PACKAGES.items():
    m = CHAMPION_RUNS[fam]["best"]
    print(
        f"  {fam:26s} | acc={m['pairwise_accuracy']:.2%} | margin={m['mean_margin']:+.3f} | "
        f"r={s1['rank']:2d} | {s1['bytes']/1e6:.2f} MB | wasmHash={s1['keccak256_wasm_hash']}"
    )

# 10. Auto-download files in Google Colab
try:
    from google.colab import files as colab_files
    print("\nStarting automatic Colab browser downloads ...")
    for fam, s1 in EXPORTED_PACKAGES.items():
        colab_files.download(s1["path"])
    colab_files.download(str(evidence_path))
    colab_files.download(str(bundle_zip))
except ImportError:
    print(f"\nNot running inside Google Colab UI. All artifacts are saved under: {EXPORT_DIR}")

# 11. Optional: Live Colab temporary Cloudflare Tunnel executor (only when toggled True)
if LAUNCH_COLAB_CLOUDFLARED_EXECUTOR:
    print("\n=== Launching Temporary Live Colab adapterexec + Cloudflare Quick Tunnel ===")
    onnx_path = download_verified("foundation_lora_fp32.onnx", PINNED_HASHES["foundation_lora_fp32.onnx"])
    engine_path = download_verified("foundation_engine.wasm", PINNED_HASHES["foundation_engine.wasm"])
    go_bin = shutil.which("go")
    if not go_bin:
        go_tgz = WORK_ROOT / "go1.25.7.linux-amd64.tar.gz"
        if not go_tgz.exists():
            req = urllib.request.Request(
                "https://go.dev/dl/go1.25.7.linux-amd64.tar.gz",
                headers={"User-Agent": "Upswing-Labs-Colab-Forge"},
            )
            with urllib.request.urlopen(req, timeout=180) as r, go_tgz.open("wb") as f:
                shutil.copyfileobj(r, f)
        subprocess.run(["tar", "-C", "/usr/local", "-xzf", str(go_tgz)], check=True)
        os.environ["PATH"] = "/usr/local/go/bin:" + os.environ.get("PATH", "")
        go_bin = "/usr/local/go/bin/go"
    cf_bin = WORK_ROOT / "cloudflared"
    if not cf_bin.exists():
        req = urllib.request.Request(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            headers={"User-Agent": "Upswing-Labs-Colab-Forge"},
        )
        with urllib.request.urlopen(req, timeout=120) as r, cf_bin.open("wb") as f:
            shutil.copyfileobj(r, f)
        cf_bin.chmod(0o755)
    env = dict(os.environ)
    env.update({
        "FOUNDATION_MODEL_PATH": str(onnx_path),
        "FOUNDATION_MODEL_HASH": PINNED_HASHES["foundation_lora_fp32.onnx"],
        "FOUNDATION_MODEL_VERSION": FOUNDATION_TAG,
        "FOUNDATION_ENGINE_WASM_PATH": str(engine_path),
        "FOUNDATION_ENGINE_WASM_HASH": PINNED_HASHES["foundation_engine.wasm"],
    })
    print("Keep this cell running while devnode validates the temporary tunnel URL.")
    exec_proc = subprocess.Popen(
        [go_bin, "run", "./cmd/adapterexec", "--package", EXPORTED_PACKAGES[primary_intent]["path"], "--listen", "127.0.0.1:8080"],
        cwd=str(KIT_DIR), env=env,
    )
    print(f"adapterexec PID={exec_proc.pid} started on 127.0.0.1:8080. Starting cloudflared tunnel ...")
    subprocess.run([str(cf_bin), "tunnel", "--url", "http://127.0.0.1:8080"], check=False)
